# Setup

In [ ]:
# set parameters
import numpy as np
import pandas as pd
pd.options.mode.chained_assignment = None  # default='warn'
import plotly.graph_objects as go
import sys
sys.path.append('../../assets/python/')
import dmg5e
import estats5e
import tfb

METADATA = {'Contributor': 'T. Dunn'}
SAVEFIGS = False

In [303]:
# functions and classes
import json
import numpy as np
import re
import uuid

COLOR_LIST = [
    '#1f77b4',  # muted blue
    '#ff7f0e',  # safety orange
    '#2ca02c',  # cooked asparagus green
    '#d62728',  # brick red
    '#9467bd',  # muted purple
    '#8c564b',  # chestnut brown
    '#e377c2',  # raspberry yogurt pink
    '#7f7f7f',  # middle gray
    '#bcbd22',  # curry yellow-green
    '#17becf',  # blue-teal
    # repeat
    '#1f77b4',  # muted blue
    '#ff7f0e',  # safety orange
    '#2ca02c',  # cooked asparagus green
    '#d62728',  # brick red
    '#9467bd',  # muted purple
    '#8c564b',  # chestnut brown
    '#e377c2',  # raspberry yogurt pink
    '#7f7f7f',  # middle gray
    '#bcbd22',  # curry yellow-green
    '#17becf',  # blue-teal
]

class MyEncoder(json.JSONEncoder):
    def default(self, o):
        return o.__dict__

class EncounterLibrary:
    def __init__(self, encounters=[], file=None):
        """
        Constructs a new encounter library.

        Parameters
        ----------
        encounters : list
            A list of encounters.
        
        file : str
            If provided then the encounters will be loaded from file.
        """
        if encounters:
            self.set_encounters(encounters)
        else:
            if file:
                self.from_json_file(file)

    def __repr__(self):
        return f'{self.__dict__}'

    def from_json_file(self, file):
        """
        Loads encounters from a JSON file.

        Parameters
        ----------
        file : str
            The name of the file.
        """
        with open(file, 'r') as fin:
            self.set_encounters(json.load(fin))
    
    def to_json_file(self, file, compact_lists=False, **kwargs):
        """
        Saves encounters as a JSON file.

        Parameters
        ----------
        file : str
            The name of the file.
        
        compact_lists : bool
            If ``True`` then lists will be written in a compact form with one entry per line.
            Default value ``False``.
        """
        s = json.dumps(self.encounters, cls=MyEncoder, **kwargs)
        if compact_lists:
            s = re.sub(r'(\[)[\s\n]+([^:,\[\]]+)(?=,|\])', r'\1\2', s)
            s = re.sub(r'(,)[\s\n]+([^:,\[\]]+)(?=,|\])', r'\1 \2', s)
            s = re.sub(r'(,[\s\n]+[^:,\[\]]+?)[\s\n]+(?=\])', r'\1', s)
        
        with open(file, 'w') as fout:
            fout.write(s)
    
    def get_encounters(self, book_paths=[], ids=None):
        """
        Returns a filtered list of encounters.

        Parameters
        ----------
        book_path : str
            A regular expression to be applied to encounter ``book_path`` attribute using the ``re.match`` function.
        
        Returns
        -------
        encounters : list
            The encounters that matched the given book_path regular expression.
        """    
        encounter_ids = ids if ids else []
        
        for book_path in book_paths:
            encounter_ids.extend([e.id for e in self.encounters if re.match(book_path, e.book_path)])
        
        return [e for e in self.encounters if e.id in encounter_ids]

    def set_encounters(self, encounters):
        self.encounters = []
        for e in encounters:
            if type(e) is dict:
                self.encounters.append(Encounter(**e))
            else:
                self.encounters.append(e)

class Encounter:
    def __init__(self, **kwargs):
        self.id = kwargs.get('id', str(uuid.uuid4()))
        self.type = kwargs.get('type', None)
        self.book_path = kwargs.get('book_path', None)
        self.allies = kwargs.get('allies', [])
        self.bystanders = kwargs.get('bystanders', [])
        self.enemies = kwargs.get('monsters', [])
        self.enemies = kwargs.get('enemies', self.enemies)

    def __repr__(self):
            return f'{self.__dict__}'

    def book(self):
        return self.book_path.split('; ')[0]

    def allies_xp_values(self):
        """
        Returns a list of XP values for the ally monsters in the encounter.

        Returns
        ----------
        xp_vals : list
            The XP values for each ally monster in the encounter.
        """
        xp_vals = []
        for monster in self.allies:
            xp_vals.extend(monster[0]*[monster[2]])
        
        return xp_vals

    def enemy_cr_values_from_xp(self):
        """
        Returns a list of CR values for the enemy monsters in the encounter.

        These CR values are determined from the monsters' XP values, which may differ from the default values for each CR.

        Returns
        ----------
        cr_vals : list
            The CR values for each enemy monster in the encounter.
        """
        def _find_nearest_loc(array, value):
            array = np.asarray(array)
            idx = (np.abs(array - value)).argmin()
            return idx
        
        cr_vals = []
        for xp in self.enemy_xp_values():
            id = _find_nearest_loc(dmg5e.MONSTER_DEFAULTS['XP'], xp)
            cr_vals.append(dmg5e.MONSTER_DEFAULTS['CR'][id])
        return cr_vals

    def enemy_monster_ids(self):
        """
        Returns a list of monster Ids for the enemy monsters in the encounter.

        Returns
        ----------
        ids : list
            The monster ids for each enemy monster in the encounter.
        """
        ids = []
        for monster in self.enemies:
            ids.extend(monster[0]*[monster[1]])
        
        return ids
    
    def enemy_xp_values(self):
        """
        Returns a list of XP values for the enemy monsters in the encounter.

        Returns
        ----------
        xp_vals : list
            The XP values for each enemy monster in the encounter.
        """
        xp_vals = []
        for monster in self.enemies:
            xp_vals.extend(monster[0]*[monster[2]])
        
        return xp_vals

    def enemy_xp_total(self):
        """
        Returns the total XP value of all enemy monsters in the encounter.

        Returns
        ----------
        xp_total : float
            The total of all enemy monster XP values in the encounter.
        """
        
        return sum(self.enemy_xp_values())

    def adjusted_enemy_xp_total(self, pc_levels, rules='2014'):
        """
        Returns the adjusted XP total for the encounter.

        Parameters
        ----------
        pc_levels : list
            The level of each PC in the encounter.
        
        rules : str
            The version of the encounter building rules used for the calculation. Default value ``'2014'``.

        Returns
        ----------
        xp_total : float
            The adjusted total XP for the encounter.
        """
        if rules == '2014':
            em = dmg5e.encounter_xp_multiplier(len(pc_levels), len(self.enemy_xp_values()))
            return em*self.enemy_xp_total()
        elif rules == '2024':
            return self.enemy_xp_total()
        elif rules == 'tfb':
            xp_vals = self.enemy_xp_values()
            xp_total = sum(xp_vals)
            xp_square = sum(np.sqrt(xp_vals))**2
            xp_multi = 0.5*(xp_square - xp_total)
            return xp_total + xp_multi*0.3

class Party:
    def __init__(self, levels=[]):
        self.levels = levels

    def average_level(self):
        """
        Returns the average level of the party.

        Returns
        -------
        level : float
            The party's average level.
        """
        return np.mean(self.levels)
    
    def xp_budget(self, rules='2014'):
        """
        Returns the adventuring day XP budget for the party.

        Parameters
        ----------
        rules : str
            The version of the encounter building rules used for the calculation. Default value ``'2014'``.

        Returns
        -------
        xp_budget : float
            The party's adventuring day XP budget.
        """
        return sum(self.pc_xp_budgets(rules))
    
    def xp_thresholds(self, rules='2014'):
        """
        Returns encounter XP thresholds for the party.

        Parameters
        ----------
        rules : str
            The version of the encounter building rules used for the calculation. Default value ``'2014'``.

        Returns
        -------
        party_xps : dict
            A dict of encounter difficulties and the party's corresponding XP thresholds.
        """
        pc_xps = self.pc_xp_thresholds(rules)
        party_xps = {}
        for diff in pc_xps:
            party_xps[diff] = sum(pc_xps[diff])
        
        return party_xps
    
    def pc_xp_budgets(self, rules='2014'):
        """
        Returns the adventuring day XP budget for each PC in the party.

        Parameters
        ----------
        rules : str
            The version of the encounter building rules used for the calculation. Default value ``'2014'``.

        Returns
        -------
        xp_budgets : list
            A list containing the adventuring day XP budgets for each PC in the party.
        """
        if rules == '2014':
            xp_budget = [300,600,1200,1700,3500,4000,5000,6000,7500,9000,10500,11500,13500,15000,18000,20000,25000,27000,30000,40000]
        else:
            xp_budget = [300,600,1200,1700,3500,4000,5000,6000,7500,9000,10500,11500,13500,15000,18000,20000,25000,27000,30000,40000]
        
        return [xp_budget[lvl-1] for lvl in self.levels]

    def pc_xp_thresholds(self, rules='2014'):
        """
        Returns encounter XP thresholds for each PC in the party.

        Parameters
        ----------
        rules : str
            The version of the encounter building rules used for the calculation. Default value ``'2014'``.

        Returns
        -------
        pc_xps : dict
            A dict of encounter difficulties, containing a lists of XP thresholds for each PC in the party.
        """
        if rules == '2014':
            xp_thresholds = {
                'Trivial':[  0,  0,   0,   0,   0,   0,   0,   0,   0,   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
                'Easy':   [ 25, 50,  75, 125, 250, 300, 350, 450, 550, 600,  800, 1000, 1100, 1250, 1400, 1600, 2000, 2100, 2400, 2800], 
                'Medium': [ 50,100, 150, 250, 500, 600, 750, 900,1100,1200, 1600, 2000, 2200, 2500, 2800, 3200, 3900, 4200, 4900, 5700], 
                'Hard':   [ 75,150, 225, 375, 750, 900,1100,1400,1600,1900, 2400, 3000, 3400, 3800, 4300, 4800, 5900, 6300, 7300, 8500], 
                'Deadly': [100,200, 400, 500,1100,1400,1700,2100,2400,2800, 3600, 4500, 5100, 5700, 6400, 7200, 8800, 9500,10900,12700], 
                'Very Deadly':  [x/2 for x in [300,600,1200,1700,3500,4000,5000,6000,7500,9000,10500,11500,13500,15000,18000,20000,25000,27000,30000,40000]], 
            }
        else:
            xp_thresholds = {
                'Low':      [ 50,100,150,250, 500, 600, 750,1000,1300,1600,1900,2200,2600,2900,3300,3800, 4500, 5000, 5500, 6400],
                'Moderate': [ 75,150,225,375, 750,1000,1300,1700,2000,2300,2900,3700,4200,4900,5400,6100, 7200, 8700,10700,13200],
                'High':     [100,200,400,500,1100,1400,1700,2100,2600,3100,4100,4700,5400,6200,7800,9800,11700,14200,17200,22000],
            }
        
        pc_xps = {}
        for diff in xp_thresholds:
            pc_xps[diff] = [xp_thresholds[diff][lvl-1] for lvl in self.levels]
        return pc_xps

def player_character_xp_budget(pc_level, rules='2014'):
    """Returns the adventuring day XP budget for a single PC of the given level.
    """
    if rules == '2014':
        XP_BUDGET = [300,600,1200,1700,3500,4000,5000,6000,7500,9000,10500,11500,13500,15000,18000,20000,25000,27000,30000,40000]
    else:
        XP_BUDGET = [300,600,1200,1700,3500,4000,5000,6000,7500,9000,10500,11500,13500,15000,18000,20000,25000,27000,30000,40000]
    return XP_BUDGET[pc_level-1]

def player_character_xp_thresholds(pc_level, rules='2014'):
    """Returns the encounter XP thresholds for each encounter difficulty for a PC of the given level.
    """
    if rules == '2014':
        XP_THRESHOLDS = {
            'Trivial':[  0,  0,   0,   0,   0,   0,   0,   0,   0,   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
            'Easy':   [ 25, 50,  75, 125, 250, 300, 350, 450, 550, 600,  800, 1000, 1100, 1250, 1400, 1600, 2000, 2100, 2400, 2800], 
            'Medium': [ 50,100, 150, 250, 500, 600, 750, 900,1100,1200, 1600, 2000, 2200, 2500, 2800, 3200, 3900, 4200, 4900, 5700], 
            'Hard':   [ 75,150, 225, 375, 750, 900,1100,1400,1600,1900, 2400, 3000, 3400, 3800, 4300, 4800, 5900, 6300, 7300, 8500], 
            'Deadly': [100,200, 400, 500,1100,1400,1700,2100,2400,2800, 3600, 4500, 5100, 5700, 6400, 7200, 8800, 9500,10900,12700], 
            'Very Deadly':  [x/2 for x in [300,600,1200,1700,3500,4000,5000,6000,7500,9000,10500,11500,13500,15000,18000,20000,25000,27000,30000,40000]], 
        }
    else:
        XP_THRESHOLDS = {
            'Low':      [ 50,100,150,250, 500, 600, 750,1000,1300,1600,1900,2200,2600,2900,3300,3800, 4500, 5000, 5500, 6400],
            'Moderate': [ 75,150,225,375, 750,1000,1300,1700,2000,2300,2900,3700,4200,4900,5400,6100, 7200, 8700,10700,13200],
            'High':     [100,200,400,500,1100,1400,1700,2100,2600,3100,4100,4700,5400,6200,7800,9800,11700,14200,17200,22000],
        }
    pc_xps = {}
    for diff in XP_THRESHOLDS:
        pc_xps[diff] = XP_THRESHOLDS[diff][pc_level-1]
    return pc_xps

def party_xp_budget(levels, rules='2014'):
    """Calculate the adventuring day XP budget for a party of PCs with the given levels.
    """
    # calculates the XP budget for a party of PCs
    return sum([player_character_xp_budget(lvl, rules=rules) for lvl in levels])

def party_xp_thresholds(levels, rules='2014'):
    """calculates the XP thresholds for a party of PCs based on their levels and the rules set being used
    """
    party_xps = {}
    for lvl in levels:
        pc_xps = player_character_xp_thresholds(lvl, rules=rules)
        for diff, xp in pc_xps.items():
            party_xps[diff] = party_xps.get(diff, 0) + xp
    
    return party_xps

def encounter_multiplier_DMG(pc_count, npc_count):
    """Returns the encounter multiplier given by the 2014 DMG
    pc_count -- number of PCs in the encounter
    npc_count -- number of NPCs in the encounter
    """
    n_array = np.asarray([1,2,3,7,11,15])
    m_array = np.asarray([0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0])
    i = 1 + n_array[n_array <= max(npc_count,1)].argmax()
    if pc_count >= 6:
        i -= 1
    elif pc_count <= 2:
        i += 1
    return m_array[i]

def encounter_difficulty(pc_thresholds, encounter_xp, rules='2014'):
    """Determines the encounter's difficulty category by comparing its XP value against the party's XP thresholds.
    """
    difficulties = list(pc_thresholds.keys())
    xp_values = list(pc_thresholds.values())
    indx = np.argsort(xp_values)
    
    if rules == '2014':
        difficulty = 'Trivial'
        for i in indx:
            if encounter_xp >= xp_values[i]:
                difficulty = difficulties[i]
    elif rules == '2024':
        difficulty = 'Very High'
        for i in reversed(indx):
            if encounter_xp <= xp_values[i]:
                difficulty = difficulties[i]
    return difficulty


def _find_nearest_loc(array, value):
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return idx

def monster_challenge_rating_from_xp(xp):
    id = _find_nearest_loc(dmg5e.MONSTER_DEFAULTS['XP'], xp)
    return dmg5e.MONSTER_DEFAULTS['CR'][id]

# Load dataset

In [304]:
# Construct encounters data frame
import numpy as np
import pandas as pd
import os

book_dict = {
    "Tyranny of Dragons": {'acronym': 'ToD', 'file': 'tod.json', 'type': 'campaign', 'date': '8/19/2014'},
    "Princes of the Apocalypse": {'acronym': 'PotA', 'file': 'pota.json', 'type': 'campaign', 'date': '4/7/2015'},
    "Curse of Strahd": {'acronym': 'CoS', 'file': 'cos.json', 'type': 'campaign', 'date': '3/16/2016'},
    "Storm King's Thunder": {'acronym': 'SKT', 'file': 'skt.json', 'type': 'campaign', 'date': '9/6/2016'},
    "Tales from the Yawning Portal": {'acronym': 'TftYP', 'file': 'tftyp.json', 'type': 'anthology', 'date': '4/4/2017'},
    "Tomb of Annihilation": {'acronym': 'ToA', 'file': 'toa.json', 'type': 'campaign', 'date': '9/19/2017'},
    "Waterdeep: Dragon Heist": {'acronym': 'W:DH', 'file': 'wdh.json', 'type': 'campaign', 'date': '9/18/2018'},
    "Waterdeep: Dungeon of the Mad Mage": {'acronym': 'W:DotMM', 'file': 'wdotmm.json', 'type': 'campaign', 'date': '11/20/2018'},
    "Ghosts of Saltmarsh": {'acronym': 'GoS', 'file': 'gos.json', 'type': 'anthology', 'date': '5/21/2019'},
    "Baldur’s Gate: Descent into Avernus": {'acronym': 'BG:DiA', 'file': 'bgdia.json', 'type': 'campaign', 'date': '9/17/2019'},
    "Icewind Dale: Rime of the Frostmaiden": {'acronym': 'ID:RotF', 'file': 'idrotf.json', 'type': 'campaign', 'date': '9/15/2020'},
    "Candlekeep Mysteries": {'acronym': 'CM', 'file': 'cm.json', 'type': 'anthology', 'date': '3/16/2021'},
    "Critical Role: Call of the Netherdeep": {'acronym': 'CR:CotN', 'file': 'cotn.json', 'type': 'campaign', 'date': '3/15/2022'},
    "Journeys through the Radiant Citadel": {'acronym': 'JttRC', 'file': 'jttrc.json', 'type': 'anthology', 'date': '7/19/2022'},
    "Dragonlance: Shadow of the Dragon Queen": {'acronym': 'D:SotDQ', 'file': 'sotdq.json', 'type': 'campaign', 'date': '12/6/2022'},
    "Keys from the Golden Vault": {'acronym': 'KftGV', 'file': 'kftgv.json', 'type': 'anthology', 'date': '2/21/2023'},
    "Phandelver and Below: The Shattered Obelisk": {'acronym': 'PaB:TSO', 'file': 'pbtso.json', 'type': 'campaign', 'date': '9/19/2023'},
    "Vecna: Eve of Ruin": {'acronym': 'V:EoR', 'file': 'veor.json', 'type': 'campaign', 'date': '5/21/2024'},
    "Quests from the Infinite Staircase": {'acronym': 'QftIS', 'file': 'qftis.json', 'type': 'anthology', 'date': '7/16/2024'},
    "Dragon Delves": {'acronym': 'DD', 'file': 'drde.json', 'type': 'anthology', 'date': '7/8/2025'},
}

# load adventures into encounter library
encounters = []
encounters_path = '../../assets/data/encounters'
for book in book_dict:
    file = book_dict[book]['file']
    elib = EncounterLibrary(file=os.path.join(encounters_path, file))
    encounters.extend(elib.encounters)

elib = EncounterLibrary(encounters=encounters)

# load campaign information
with open('../../assets/data/encounters/campaigns.json', 'r') as fin:
    campaigns = json.load(fin)

encounters = []
for campaign in campaigns:
    for group in campaign['groups']:
        encs = elib.get_encounters(ids=group['encounter_ids'])
        party = Party(group['party'])
        for enc in encs:
            if enc.type != 'combat': continue
            e = {}
            e['id'] = enc.id
            e['type'] = enc.type
            e['book'] = enc.book()
            e['book_acronym'] = book_dict.get(enc.book(), {}).get('acronym', '')
            e['adventure'] = enc.book_path.split('; ')[1]
            e['monsters'] = enc.enemy_monster_ids()
            e['monsters_xp'] = enc.enemy_xp_values()
            #e['monsters_cr'] = [monster_challenge_rating_from_xp(xp) for xp in enc.enemy_xp_values()]
            e['monsters_cr'] = enc.enemy_cr_values_from_xp()
            e['n_monsters'] = len(enc.enemy_xp_values())
            e['n_unique_monsters'] = len(np.unique(enc.enemy_monster_ids()))
            e['party'] = party.levels
            e['XP_total'] = enc.enemy_xp_total()
            e['PC_XP_total'] = e['XP_total']/len(party.levels)
            try:
                e['2014 adj_XP_total'] = enc.adjusted_enemy_xp_total(party.levels, rules='2014')
                e['2014 party_xp_thresholds'] = party_xp_thresholds(party.levels, rules='2014')
                e['2014 party_xp_budget'] = party_xp_budget(party.levels, rules='2014')
                e['2014 difficulty'] = encounter_difficulty(e['2014 party_xp_thresholds'], e['2014 adj_XP_total'], rules='2014')

                e['2024 adj_XP_total'] = enc.adjusted_enemy_xp_total(party.levels, rules='2024')
                e['2024 party_xp_thresholds'] = party_xp_thresholds(party.levels, rules='2024')
                e['2024 party_xp_budget'] = party_xp_budget(party.levels, rules='2024')
                e['2024 difficulty'] = encounter_difficulty(e['2024 party_xp_thresholds'], e['2024 adj_XP_total'], rules='2024')

                e['TFB adj_XP_total'] = enc.adjusted_enemy_xp_total(party.levels, rules='tfb')
                e['TFB party_xp_thresholds'] = party_xp_thresholds(party.levels, rules='2014')
                e['TFB party_xp_budget'] = party_xp_budget(party.levels, rules='2014')
                e['TFB difficulty'] = encounter_difficulty(e['TFB party_xp_thresholds'], e['TFB adj_XP_total'], rules='2014')
                encounters.append(e)
            except:
                #print(f"{enc['book_path']}:")
                #print(f"  monsters: {enc['monsters']}")
                pass

print(f'total encounters: {len(encounters)}')

dfe = pd.DataFrame({
    'id': [e['id'] for e in encounters],
    'book': [e['book'] for e in encounters],
    'book_acronym': [e['book_acronym'] for e in encounters],
    'adventure': [e['adventure'] for e in encounters],
    'type': [e['type'] for e in encounters],
    'party': [e['party'] for e in encounters],
    'party_level': [np.mean(e['party']) for e in encounters],
    'monsters': [e['monsters'] for e in encounters],
    'monsters_xp': [e['monsters_xp'] for e in encounters],
    'monsters_cr': [e['monsters_cr'] for e in encounters],
    'n_monsters': [e['n_monsters'] for e in encounters],
    'n_unique_monsters': [e['n_unique_monsters'] for e in encounters],
    'XP_total': [e['XP_total'] for e in encounters],
    'PC_XP_total': [e['PC_XP_total'] for e in encounters],

    '2014 adj_XP_total': [e['2014 adj_XP_total'] for e in encounters],
    '2014 party_XP_budget': [e['2014 party_xp_budget'] for e in encounters],
    '2014 difficulty': [e['2014 difficulty'] for e in encounters],

    '2024 adj_XP_total': [e['2024 adj_XP_total'] for e in encounters],
    '2024 party_XP_budget': [e['2024 party_xp_budget'] for e in encounters],
    '2024 difficulty': [e['2024 difficulty'] for e in encounters],

    'TFB adj_XP_total': [e['TFB adj_XP_total'] for e in encounters],
    'TFB party_XP_budget': [e['TFB party_xp_budget'] for e in encounters],
    'TFB difficulty': [e['TFB difficulty'] for e in encounters],
})
book_categories = list(book_dict.keys())
book_acronym_categories = [v['acronym'] for v in book_dict.values()]
dfe['book'] = dfe['book'].astype('category')
dfe['book'] = dfe['book'].cat.set_categories(book_categories, ordered=True)
dfe['book_acronym'] = dfe['book_acronym'].astype('category')
dfe['book_acronym'] = dfe['book_acronym'].cat.set_categories(book_acronym_categories, ordered=True)
dfe['2014 difficulty'] = dfe['2014 difficulty'].astype('category')
dfe['2014 difficulty'] = dfe['2014 difficulty'].cat.set_categories(['Trivial','Easy','Medium','Hard', 'Deadly', 'Very Deadly'], ordered=True)
dfe['2014 XP_ratio'] = 2*dfe['2014 adj_XP_total']/dfe['2014 party_XP_budget']
dfe['2014 XP_mult'] = dfe['2014 adj_XP_total']/dfe['XP_total']
dfe['2024 difficulty'] = dfe['2024 difficulty'].astype('category')
dfe['2024 difficulty'] = dfe['2024 difficulty'].cat.set_categories(['Low','Moderate','High','Very High'], ordered=True)
dfe['2024 XP_ratio'] = 2*dfe['2024 adj_XP_total']/dfe['2024 party_XP_budget']
dfe['2024 XP_mult'] = dfe['2024 adj_XP_total']/dfe['XP_total']
dfe['TFB difficulty'] = dfe['TFB difficulty'].astype('category')
dfe['TFB difficulty'] = dfe['TFB difficulty'].cat.set_categories(['Trivial','Easy','Medium','Hard', 'Deadly', 'Very Deadly'], ordered=True)
dfe['TFB XP_ratio'] = 2*dfe['TFB adj_XP_total']/dfe['TFB party_XP_budget']
dfe['TFB XP_mult'] = dfe['TFB adj_XP_total']/dfe['XP_total']

total encounters: 3758


In [305]:
# load monster data
ABILITY_MOD_COLUMNS = ['Str Mod','Dex Mod','Con Mod','Int Mod','Wis Mod','Cha Mod',]
SAVE_BONUS_COLUMNS = ['Str Save','Dex Save','Con Save','Int Save','Wis Save','Cha Save',]
CONDITION_COLUMNS = ['Blinded','Charmed','Deafened','Exhaustion','Frightened','Grappled','Incapacitated',
    'Invisible','Paralyzed','Petrified','Poisoned','Prone','Restrained','Stunned','Unconscious',]
DAMAGE_COLUMNS = ['Bludgeoning','Piercing','Slashing','Acid','Cold','Fire','Force','Lightning','Necrotic','Poison','Psychic','Radiant','Thunder',]

# load monster data
df1 = pd.read_csv('../../assets/data/monsters-2014.csv')
df2 = pd.read_csv('../../assets/data/monsters-2024.csv')
dfm = pd.concat([df1, df2])

#dfm['adj HP'] = dfm.apply(lambda row: unadjust_hit_points(row), axis=1)

dfm['Type'] = dfm['Type'].str.title()
dfm['Size'] = dfm['Size'].str.title()
dfm = dfm.astype({'Book': 'category', 'Size': 'category', 'Type': 'category'})

dfm['Size'] = dfm['Size'].cat.set_categories(['Tiny','Small','Medium','Large','Huge','Gargantuan'], ordered=True)

dfm['DC'] = dfm['Save DC']
dfm['adj DC'] = dfm['DC']
dfm['eAC']  = (dfm['adj AC'] + dfm['adj SB'] + 14)/2
dfm['eHP']  = dfm.apply(lambda row: estats5e.effHP(row['adj HP'], row['eAC']), axis=1)
dfm['eDPR'] = dfm.apply(lambda row: estats5e.effDPR(row['adj DPR'], row['adj AB']), axis=1)
dfm['eXP']  = dfm.apply(lambda row: estats5e.effXP(row['adj HP'], row['eAC'], row['adj DPR'], row['adj AB']), axis=1)

dfm

,Monster,Monster ID,Book,Page,Type,Size,Category,Legendary,CR,PB,...,adj AB,DPR,adj DPR,Save DC,DC,adj DC,eAC,eHP,eDPR,eXP
0,Aarakocra,17100-aarakocra,MM (2014),12.0,Humanoid,Medium,generic,N,0.250,2,...,4.0,5.500000,6.666666,NaN,NaN,NaN,13.25,18.354755,5.788287,26.394230
1,Aarakocra Simulacrum,300122-aarakocra-simulacrum,SKT,188.0,Humanoid,Medium,generic,N,0.125,2,...,4.0,5.500000,6.666666,NaN,NaN,NaN,13.25,8.157669,5.788287,11.730769
2,Aartuk Elder,2821141-aartuk-elder,SAiS:BAM,8.0,Plant,Large,generic,N,3.000,2,...,6.0,22.000000,22.000000,NaN,NaN,NaN,15.85,120.576074,21.830113,629.855769
3,Aartuk Starhorror,2821142-aartuk-starhorror,SAiS:BAM,9.0,Plant,Medium,generic,N,2.000,2,...,3.0,16.000000,16.000000,NaN,NaN,NaN,14.60,77.397674,12.899612,249.600000
4,Aartuk Weedling,2821143-aartuk-weedling,SAiS:BAM,9.0,Plant,Medium,generic,N,2.000,2,...,4.0,16.000000,16.000000,NaN,NaN,NaN,14.90,58.406048,13.891890,200.200000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
720,Zhentilar Soldier,5900685-zhentilar-soldier,FRAiF,285.0,Humanoid,Medium,generic,N,1.000,2,...,4.0,16.000000,16.000000,12.0,12.0,12.0,14.25,32.010503,13.891890,110.000000
721,Zlan,5900686-zlan,FRAiF,287.0,Undead,Huge,unique,L,18.000,6,...,11.0,207.000000,207.000000,19.0,19.0,19.0,24.25,697.444927,269.589496,38318.387019
722,Zombie,4775851-zombie,MM (2024),346.0,Undead,Medium,generic,N,0.250,2,...,3.0,5.500000,5.500000,NaN,NaN,NaN,10.60,24.348973,4.434242,26.992308
723,Zombie Clot,6491405-zombie-clot,RTHW,283.0,Undead,Huge,generic,N,6.000,3,...,8.0,32.166668,39.166668,16.0,16.0,16.0,14.00,205.372897,43.722245,2161.698792


In [306]:
monster_decoder = {
    '266705-preeta-kreepa': '16947-mage',
    '280435-urstul-floxin': '16790-assassin',
    '1528966-hag-of-the-fetid-gaze': '16911-green-hag', #'301746-green-hag-coven-variant',
    '190986-keresta-delvingstone': '17043-vampire',
    '301254-zegdar': '16958-minotaur',
    '277614-vertrand-shadowdusk': '16789-archmage',
    '273045-animated-statue': '16789-archmage',
    '301411-swarm-of-insects-centipedes': '17029-swarm-of-insects',
    '301511-gray-slaad-control-gem-variant': '17114-gray-slaad',
    '299425-snow-maiden': '17017-specter',
    '17372-the-abbot': '16840-deva',
    '1123098-piercer-ice-variant': '17191-piercer',
    '273037-violence': '16947-mage',
    '27711-half-green-dragon-assassin': '16790-assassin',
    '280305-roscoe-underbough': '17055-wererat',
    '285389-havia-quickknife': '17337-martial-arts-adept',
    '300034-duchess-brimskarda': '16862-fire-giant',
    '286176-terracotta-warrior': '16786-animated-armor',
    '276570-intelligent-black-pudding': '16808-black-pudding',
    '280299-grumshar': '17325-apprentice-wizard',
    '285356-manafret-cherryport': '16947-mage',
    '273036-turbulence': '16947-mage',
    '276630-gorka-tharn': '16962-mummy-lord',
    '277050-fazrian': '16980-planetar',
    '285289-kalain': '17328-bard',
    '1123104-kadroth': '16836-cult-fanatic',
    '267448-wyllow': '17326-archdruid',
    '301414-swarm-of-insects-spiders': '17029-swarm-of-insects',
    '285362-yorn-the-terror': '17035-thug',
    '285351-swarm-of-mechanical-spiders': '17029-swarm-of-insects',
    '267516-undead-bulette': '16818-bulette',
    '275719-crystal-golem': '17025-stone-golem',
    '277609-derrion-shadowdusk': '17330-champion',
    '1123106-cult-fanatic-knights-of-the-black-sword': '16836-cult-fanatic',
    '299961-cinderhild': '16969-ogre',
    '1123120-reghed-warrior': '17038-tribal-warrior',
    '295011-sentient-gray-ooze': '16909-gray-ooze',
    '300551-lord-khaspere-drylund': '16966-noble',
    '300501-imperator-uthor': '17026-storm-giant',
    '3805745-goblin-boss-variant': '17155-goblin-boss',
    '266134-shunn-shurreth': '17134-drow-elite-warrior',
    '295010-sentient-ochre-jelly': '16967-ochre-jelly',
    '299915-zaltember': '17185-half-ogre',
    '277605-melissara-shadowdusk': '16789-archmage',
    '301415-swarm-of-insects-wasps': '17029-swarm-of-insects',
    '17141-cave-bear': '16983-polar-bear',
    '299572-animated-halberd': '16865-flying-sword',
    '277622-zalthar-shadowdusk': '17128-death-knight',
    '27708-half-blue-dragon-gladiator': '16903-gladiator',
    '286174-tomb-guardian': '16863-flesh-golem',
    '277124-valtagar-steelshadow': '16789-archmage',
    '277616-berlain-shadowdusk': '16789-archmage',
    '285275-obliteros': '16894-giant-shark',
    '301694-bone-devil-polearm-variant': '16813-bone-devil',
    '274991-deformed-duergar': '16850-duergar',
    '280438-yalah-gralhund': '16966-noble',
    '300498-princess-serissa': '17026-storm-giant',
    '272306-torbit': '16790-assassin',
    '285943-yuan-ti-priest': '301425-yuan-ti-malison-type-3',
    '266680-marta-moonshadow': '16947-mage',
    '17350-iymrith': '16776-ancient-blue-dragon',
    '265953-shadowy-duplicate': '17010-shadow',
    '1123112-lonelywood-banshee': '17089-banshee',
    '285953-sekelok': '17330-champion',
    '276622-stonecloak': '17025-stone-golem',
    '267381-darribeth-meltimer': '16947-mage',
    '300554-pow-ming': '16947-mage',
    '265945-harria-valashtar': '16799-bandit-captain',
    '277158-stalagma-steelshadow': '16772-adult-silver-dragon',
    '273059-nester': '16789-archmage',
    '656065-umber-hulk-modified-variant': '17205-umber-hulk',
    '285894-bag-of-nails': '16790-assassin',
    '275607-thwad-underbrew': '17330-champion',
    '275796-umbraxakar': '16767-adult-bronze-dragon',
    '265939-uktarl-krannoc': '16799-bandit-captain',
    '266127-trenzia': '17091-flameskull',
    '285375-kaevja-cynavern': '16947-mage',
    '1128460-cultist-knights-of-the-black-sword': '16835-cultist',
    '266698-nerozar-the-defeated': '17111-beholder-zombie',
    '274927-drivvin-freth': '16789-archmage',
    '417128-madcap': '2556141-redcap',
    '280387-soluun-xibrindas': '149812-drow-gunslinger',
    '299436-kiril-stoyanovich': '17057-werewolf',
    '276644-hammer-handed-golem': '17025-stone-golem',
    '300475-cressaro': '16827-cloud-giant',
    '1123130-tribal-warrior-spore-servants': '17038-tribal-warrior',
    '1123128-telepathic-pentacle': '16929-hydra',
    '285396-mookie-plush': '17337-martial-arts-adept',
    '280407-noska-urgray': '17035-thug',
    '274961-zox-clammersham': '16789-archmage',
    '285864-clay-gladiator': '16903-gladiator',
    '288137-durnn': '16925-hobgoblin',
    '356626-rose-durst': '16871-ghost',
    '356633-thorn-durst': '16871-ghost',

    '294423-thaxalia': '17099-beholder', # reduced threat beholder
    '294830-reduced-threat-owlbear': '16975-owlbear',
    '294798-reduced-threat-wight': '17059-wight',
    '294518-reduced-threat-flesh-golem': '16863-flesh-golem',
    '294800-reduced-threat-aboleth': '16762-aboleth',
    '294808-reduced-threat-otyugh': '16973-otyugh',
    '294792-reduced-threat-hook-horror': '17162-hook-horror',
    '294537-reduced-threat-carrion-crawler': '17138-carrion-crawler',
    '294872-reduced-threat-stone-golem': '17025-stone-golem',
    '294839-reduced-threat-ochre-jelly': '16967-ochre-jelly',
    '294794-reduced-threat-wyvern': '17065-wyvern',
    '294868-reduced-threat-helmed-horror': '17159-helmed-horror',
    '294529-reduced-threat-basilisk': '16801-basilisk',
    '294512-reduced-threat-vrock': '17047-vrock',
    '294536-reduced-threat-ettercap': '16859-ettercap',
    '294534-reduced-threat-darkmantle': '16837-darkmantle',
    '294396-reduced-threat-hezrou': '16922-hezrou',
    '294538-reduced-threat-behir': '16804-behir',
    '294525-reduced-threat-remorhaz': '16995-remorhaz',
    '294524-reduced-threat-glabrezu': '16902-glabrezu',
    '294858-reduced-threat-dragon-turtle': '16845-dragon-turtle',
    '294827-reduced-threat-peryton': '17190-peryton',
    '294811-reduced-threat-displacer-beast': '17130-displacer-beast',
    '294522-reduced-threat-clay-golem': '16825-clay-golem',
    '294833-reduced-threat-black-pudding': '16808-black-pudding',

    '782417-failed-dragonpriest': '17040-troll',
    '294945-four-armed-gargoyle': '16868-gargoyle',
    '295770-chief-nosnra': '16867-frost-giant',
    '1198558-swarm-of-rats-diseased-variant': '17032-swarm-of-rats',
    '275578-carrion-ogre': '16969-ogre',
    '276626-mad-golem': '17025-stone-golem',
    '266150-gelatinous-cube-giant-ooze-variant': '16869-gelatinous-cube', # unique monster, W:DotMM
    '296882-elder-black-pudding': '16808-black-pudding', # should be unique, cr is not right, TftYP

    '301746-green-hag-coven-variant': '16911-green-hag', # unique monster, MM 2014
    '301755-night-hag-coven-variant': '16965-night-hag', # unique monster, MM 2014
    '301760-sea-hag-coven-variant': '17008-sea-hag', # unique monster, MM 2014
    #'301418-yuan-ti-malison-type-1': '', # unique monster, MM 2014
    #'301421-yuan-ti-malison-type-2': '', # unique monster, MM 2014
    #'301425-yuan-ti-malison-type-3': '', # unique monster, MM 2014


    #'shockerstomper': '', # unique monster, WDotMM
}

monsters = []
for e in encounters:
    for m, cr in zip(e['monsters'], e['monsters_cr']):
        if cr > 0:
            monsters.append(monster_decoder.get(m, m))
monsters = list(set(monsters))
print(len(monsters))

db_monsters = dfm['Monster ID'].unique()
print(len(db_monsters))
missing_monsters = []
for m in monsters:
    if m in db_monsters: continue
    missing_monsters.append(m)
    print(m)
print(len(missing_monsters))

949
2823
shockerstomper
1


In [307]:
monsters = []
for e in encounters:
    for m, cr in zip(e['monsters'], e['monsters_cr']):
        if cr > 0:
            monsters.append(monster_decoder.get(m, m))
monsters = monsters

dfm = dfm.set_index('Monster ID')
dfm['book appearances'] = 0
for monster in dfm.index:
    dfm.loc[monster, 'book appearances'] = len([m for m in monsters if m == monster])
dfm['book appearances']
dfm = dfm.reset_index()

In [308]:
# encounter monsters dataframe
data = {
    'party_level': [],
    '2014 difficulty': [],
    'Monster ID': [],
    'Encounter ID': [],
    'XP ratio': [],
    'encounter_monsters': [],
}
for e in encounters:
    pc_xp_budget = party_xp_budget(e['party'], rules='2014')/(2*len(e['party']))
    monsters = [monster_decoder.get(m, m) for m in e['monsters']]
    data['party_level'] += [np.mean(e['party'])]*len(monsters)
    data['2014 difficulty'] += [e['2014 difficulty']]*len(monsters)
    data['Monster ID'] += [monster_decoder.get(m, m) for m in e['monsters']]
    data['Encounter ID'] += [e['id']]*len(monsters)
    data['XP ratio'] += [xp/pc_xp_budget for xp in e['monsters_xp']]
    data['encounter_monsters'] += [e['n_monsters']]*len(monsters)


dfem = pd.DataFrame(data)
dfem = pd.merge(dfem, dfm, on='Monster ID', how='inner')
dfem

,party_level,2014 difficulty,Monster ID,Encounter ID,XP ratio,encounter_monsters,Monster,Book,Page,Type,...,DPR,adj DPR,Save DC,DC,adj DC,eAC,eHP,eDPR,eXP,book appearances
0,1.0,Easy,17095-twig-blight,756de097-2858-4295-82a7-a447be3fb5c5,0.166667,4,Twig Blight,MM (2014),32.0,Plant,...,3.5,3.5,NaN,NaN,NaN,12.85,5.946511,2.821790,4.194952,89
1,1.0,Easy,17095-twig-blight,756de097-2858-4295-82a7-a447be3fb5c5,0.166667,4,Twig Blight,MM (2014),32.0,Plant,...,3.5,3.5,NaN,NaN,NaN,12.85,5.946511,2.821790,4.194952,89
2,1.0,Easy,17095-twig-blight,756de097-2858-4295-82a7-a447be3fb5c5,0.166667,4,Twig Blight,MM (2014),32.0,Plant,...,3.5,3.5,NaN,NaN,NaN,12.85,5.946511,2.821790,4.194952,89
3,1.0,Easy,17095-twig-blight,756de097-2858-4295-82a7-a447be3fb5c5,0.166667,4,Twig Blight,MM (2014),32.0,Plant,...,3.5,3.5,NaN,NaN,NaN,12.85,5.946511,2.821790,4.194952,89
4,1.0,Easy,16891-giant-rat,05538902-bb6e-4a98-b0d5-88c68e2eb1e9,0.166667,3,Giant Rat,MM (2014),327.0,Beast,...,4.5,4.5,NaN,NaN,NaN,12.40,8.949583,4.186172,9.328846,67
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14954,20.0,Hard,4468331-mirror-shade,f169b01a-b1a4-4a69-9f37-e096ea40610d,0.295000,4,Mirror Shade,VEoR,226.0,Undead,...,57.0,57.0,16.0,16.0,16.0,18.90,259.170578,67.164809,3875.287500,4
14955,20.0,Hard,4468331-mirror-shade,f169b01a-b1a4-4a69-9f37-e096ea40610d,0.295000,4,Mirror Shade,VEoR,226.0,Undead,...,57.0,57.0,16.0,16.0,16.0,18.90,259.170578,67.164809,3875.287500,4
14956,20.0,Hard,4468331-mirror-shade,f169b01a-b1a4-4a69-9f37-e096ea40610d,0.295000,4,Mirror Shade,VEoR,226.0,Undead,...,57.0,57.0,16.0,16.0,16.0,18.90,259.170578,67.164809,3875.287500,4
14957,20.0,Hard,4468331-mirror-shade,f169b01a-b1a4-4a69-9f37-e096ea40610d,0.295000,4,Mirror Shade,VEoR,226.0,Undead,...,57.0,57.0,16.0,16.0,16.0,18.90,259.170578,67.164809,3875.287500,4


# Figures

In [318]:
# Fig. 1: Plots a histogram of monsters per encounter
import plotly.graph_objects as go
from plotly.subplots import make_subplots

levels = list(range(1,21))

fig = go.Figure()

yMed = []
yHigh = []
yLow  = []
difficulties = ['Easy','Medium','Hard','Deadly','Very Deadly']
#difficulties = ['Hard','Deadly']
for level in levels:
    #dft = dfe[dfe['party_level'].eq(level) & dfe['2014 XP_ratio'].ge(0.30)]
    dft = dfe[dfe['party_level'].eq(level) & dfe['2014 difficulty'].isin(difficulties)]
    monster_cr_values = []
    for x in dft['monsters_cr']:
        monster_cr_values.extend(x)

    yMed.append(np.median(monster_cr_values))
    yHigh.append(np.quantile(monster_cr_values, 0.8))
    yLow.append(np.quantile(monster_cr_values, 0.2))


fig.add_trace(go.Scatter(
    x=levels,
    y=yLow,
    showlegend=False,
    mode='lines',
    line_width=0,
    line_color=COLOR_LIST[0],
    hoverinfo='skip',
))

fig.add_trace(go.Scatter(
    x=levels,
    y=yHigh,
    showlegend=False,
    mode='lines',
    line_width=0,
    fill='tonexty',
    line_color=COLOR_LIST[0],
    hoverinfo='skip',
))

fig.add_trace(go.Scatter(
    x=levels,
    y=yMed,
    showlegend=False,
    mode='lines',
    line_color=COLOR_LIST[0],
    #customdata=dfC.index,
    hovertemplate = ''
        + 'level %{x:.0f}<br>'
        + 'CR %{y:.0f}'
        + '<extra></extra>'
))

fig.add_trace(go.Scatter(
    x=[0,21],
    y=[0,21],
    showlegend=False,
    mode='lines',
    line_color='black',
    line_dash='dash',
    hoverinfo='skip',
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', range=[0,21], tickformat='.0f', dtick=5, minor_dtick=1),
    yaxis=dict(title_text='challenge rating', range=[0,21], tickformat='.0f', dtick=5, minor_dtick=1),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
if SAVEFIGS:
    fig.write_image('../../assets/images/adventure-encounter-monsters.png')
    fig.update_layout(autosize=True, width=None, height=None)
    tfb.save_fig_html(fig, format='large', name=f'./fig-monster-cr-range-vs-level-large', selector={'name': 'none'})
    tfb.save_fig_html(fig, format='small', name=f'./fig-monster-cr-range-vs-level-small', selector={'name': 'none'})

In [310]:
# Fig. 2: Plots a histogram of monsters per encounter
import plotly.graph_objects as go
from plotly.subplots import make_subplots

levels = list(range(1,21))

fig = go.Figure()

yMed = []
yHigh = []
yLow  = []
difficulties = ['Easy','Medium','Hard','Deadly','Very Deadly']
for level in levels:
    #dft = dfe[dfe['party_level'].eq(level) & dfe['2014 XP_ratio'].ge(0.30)]
    dft = dfe[dfe['party_level'].eq(level) & dfe['2014 difficulty'].isin(difficulties)]
    monster_xp_values = []
    for x in dft['monsters_xp']:
        monster_xp_values.extend(x)

    yMed.append(np.median(monster_xp_values)/(0.5*player_character_xp_budget(level)))
    yHigh.append(np.quantile(monster_xp_values, 0.8)/(0.5*player_character_xp_budget(level)))
    yLow.append(np.quantile(monster_xp_values, 0.2)/(0.5*player_character_xp_budget(level)))

fig.add_trace(go.Scatter(
    x=levels,
    y=yLow,
    showlegend=False,
    mode='lines',
    line_width=0,
    line_color=COLOR_LIST[0],
    hoverinfo='skip',
))

fig.add_trace(go.Scatter(
    x=levels,
    y=yHigh,
    showlegend=False,
    mode='lines',
    line_width=0,
    fill='tonexty',
    line_color=COLOR_LIST[0],
    hoverinfo='skip',
))

fig.add_trace(go.Scatter(
    x=levels,
    y=yMed,
    showlegend=False,
    mode='lines',
    line_color=COLOR_LIST[0],
    #customdata=dfC.index,
    hovertemplate = ''
        + 'level %{x:.0f}<br>'
        + 'XP ratio %{y:.3f}'
        + '<extra></extra>'
))

fig.add_trace(go.Scatter(
    x=[1,20],
    y=[1.2,1.2],
    showlegend=False,
    mode='lines',
    line_dash='dash',
    line_color='black',
    hoverinfo='skip',
))


# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', range=[0,21], tickformat='.0f', dtick=5, minor_dtick=1),
    yaxis=dict(title_text='monster XP ratio', range=[0,1.5], tickformat='.1f'),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
if SAVEFIGS:
    fig.update_layout(autosize=True, width=None, height=None)
    tfb.save_fig_html(fig, format='large', name=f'./fig-monster-xp-range-vs-level-large', selector={'name': 'none'})
    tfb.save_fig_html(fig, format='small', name=f'./fig-monster-xp-range-vs-level-small', selector={'name': 'none'})

In [311]:
# Fig. 3: median monster XP by encounter difficulty vs level
import plotly.graph_objects as go

stat = 'XP ratio'

groups = [
    #{'name': 'Trivial encounters', 'difficulties': ['Trivial']},
    {'name': 'Easy encounters', 'difficulties': ['Easy']},
    {'name': 'Medium encounters', 'difficulties': ['Medium']},
    {'name': 'Hard encounters', 'difficulties': ['Hard']},
    {'name': 'Deadly encounters', 'difficulties': ['Deadly']},
    #{'name': 'Very Deadly encounters', 'difficulties': ['Very Deadly']},
]

fig = go.Figure()

colors = iter(COLOR_LIST)
for group in groups:
    levels = []
    values = []
    for lvl in range(1,21):
        monsters = []
        for e in encounters:
            if np.mean(e['party']) != lvl: continue
            if e['2014 difficulty'] not in group['difficulties']: continue
            for m, cr in zip(e['monsters'], e['monsters_cr']):
                if cr > 0:
                    monsters.append(monster_decoder.get(m, m))
        monsters = monsters
        if len(monsters) < 4: continue
        
        dft = dfm.set_index('Monster ID')
        dft = dft[dft['eXP'].gt(0)]
        dft['book appearances'] = 0
        for monster in dft.index:
            dft.loc[monster, 'book appearances'] = len([m for m in monsters if m == monster])
        
        levels.append(lvl)
        values.append(sum(dft['book appearances']*dft['eXP'])/(sum(dft['book appearances'])*0.5*player_character_xp_budget(lvl)))


    tfb.plot_data_and_fit(fig, 
        x=levels,
        y=values,
        line_color=next(colors), 
        name=group['name'],
        legendgroup=group['name'],
    )

fig.add_trace(go.Scatter(
    x=[1,20],
    y=[1.2,1.2],
    showlegend=False,
    mode='lines',
    line_dash='dash',
    line_color='black',
    hoverinfo='skip',
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', automargin=True, range=[0,21], dtick=5, minor_dtick=1),
    yaxis=dict(title_text=stat, range=[0,2], dtick=0.2, minor_dtick=0.1),
    legend=dict(xanchor='left', x=0.00, yanchor='top', y=1.00),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
if SAVEFIGS:
    fig.update_layout(autosize=True, width=None, height=None)
    tfb.save_fig_html(fig, format='large', name=f'./fig-monster-xp-vs-level-by-difficulty-large')
    tfb.save_fig_html(fig, format='small', name=f'./fig-monster-xp-vs-level-by-difficulty-small')

In [312]:
# Fig. 4: Plots the average number of monsters per encounter by level for each difficulty
import plotly.graph_objects as go

levels = list(range(1,21))

groups = [
    #{'name': 'Trivial encounters', 'difficulties': ['Trivial']},
    {'name': 'Easy encounters', 'difficulties': ['Easy']},
    {'name': 'Medium encounters', 'difficulties': ['Medium']},
    {'name': 'Hard encounters', 'difficulties': ['Hard']},
    {'name': 'Deadly encounters', 'difficulties': ['Deadly']},
    #{'name': 'Very Deadly encounters', 'difficulties': ['Very Deadly']},
]

fig = go.Figure()
colors = iter(COLOR_LIST)
for group in groups:
    dft = dfe[dfe['2014 difficulty'].isin(group['difficulties'])]
    dft = dft[['party_level','n_monsters']].groupby(['party_level']).mean().reset_index()

    tfb.plot_data_and_fit(fig, 
        x=dft['party_level'],
        y=dft['n_monsters'],
        line_color=next(colors), 
        name=group['name'],
        legendgroup=group['name'],
    )

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', range=[0,21], tickformat='.0f', dtick=5, minor_dtick=1),
    yaxis=dict(title_text='monsters per encounter', range=[0,10], tickformat='.0f', dtick=2, minor_dtick=1),
    legend=dict(xanchor='left', x=0.00, yanchor='top', y=1.00),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
if SAVEFIGS:
    fig.update_layout(autosize=True, width=None, height=None)
    tfb.save_fig_html(fig, format='large', name=f'./fig-monsters-per-encounter-vs-level-by-difficulty-large')
    tfb.save_fig_html(fig, format='small', name=f'./fig-monsters-per-encounter-vs-level-by-difficulty-small')

In [313]:
# Fig. 5: distribution of encounter size for each difficulty
import plotly.graph_objects as go

groups = [
    #{'name': 'Trivial encounters', 'difficulties': ['Trivial']},
    {'name': 'Easy encounters', 'difficulties': ['Easy']},
    {'name': 'Medium encounters', 'difficulties': ['Medium']},
    {'name': 'Hard encounters', 'difficulties': ['Hard']},
    {'name': 'Deadly encounters', 'difficulties': ['Deadly']},
    #{'name': 'Very Deadly encounters', 'difficulties': ['Very Deadly']},
]

fig = go.Figure()

for group in groups:
    name = group['name']
    df1 = dfe[dfe['2014 difficulty'].isin(group['difficulties'])]
    y, x = np.histogram(df1['n_monsters'], bins=np.linspace(1,21,21)-0.5)
    x = (x[1:] + x[0:-1])/2
    fig.add_trace(go.Scatter(
        x=x, 
        y=y/sum(y),
        #y=[d5e.encounter_multiplier_DMG(4, xx)*yy/sum(y) for xx, yy in zip(x,y)],
        mode='markers+lines',
        name=name,
        showlegend=True,
        hovertemplate = f'<b>{name}</b><br>'
                + 'monsters %{x:.0f}<br>'
                + 'probability %{y:.1%}'
                + '<extra></extra>'
    ))

y, x = np.histogram(dfe['n_monsters'], bins=np.linspace(1,21,21)-0.5)
x = (x[1:] + x[0:-1])/2
fig.add_trace(go.Scatter(
    x=x, 
    y=y/sum(y),
    mode='markers+lines',
    name='all encounters',
    line_color='black',
    line_dash='dash',
    showlegend=True,
    hovertemplate = f'<b>all encounters</b><br>'
                    + 'monsters %{x:.0f}<br>'
                    + 'probability %{y:.1%}'
                    + '<extra></extra>'
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='monsters per encounter', range=[0,10], dtick=1, minor_dtick=1),
    yaxis=dict(title_text='encounters [%]', range=[-0.01, 0.35], dtick=0.05, minor_dtick=0.05, tickformat='.0%'),
    legend=dict(
        xanchor='right', x=1.00, 
        yanchor='top', y=1.00,
        orientation='v',
    ),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

if SAVEFIGS:
    fig.update_layout(autosize=True, width=None, height=None)
    tfb.save_fig_html(fig, format='large', name=f'./fig-monsters-per-encounter-by-difficulty-large', style='width: 600px;')
    tfb.save_fig_html(fig, format='small', name=f'./fig-monsters-per-encounter-by-difficulty-small', style='width: 600px;')

In [314]:
# Fig. 6: monster adjusted AB vs level for each difficulty
import plotly.graph_objects as go

stat = 'adj AB'

groups = [
    #{'name': 'Trivial encounters', 'difficulties': ['Trivial']},
    {'name': 'Easy encounters', 'difficulties': ['Easy']},
    {'name': 'Medium encounters', 'difficulties': ['Medium']},
    {'name': 'Hard encounters', 'difficulties': ['Hard']},
    {'name': 'Deadly encounters', 'difficulties': ['Deadly']},
    #{'name': 'Very Deadly encounters', 'difficulties': ['Very Deadly']},
]

fig = go.Figure()

colors = iter(COLOR_LIST)
for group in groups:
    dft = dfem[dfem['2014 difficulty'].isin(group['difficulties'])]
    dft = dft[['party_level',stat]].groupby(['party_level']).mean().reset_index()

    tfb.plot_data_and_fit(fig, 
        x=dft['party_level'],
        y=dft[stat],
        line_color=next(colors), 
        name=group['name'],
        legendgroup=group['name'],
        hovertemplate='<b>' + group['name'] + '</b><br>level %{x:.0f}<br>AB %{y:.1f}<extra></extra>',
    )

dfG = dfm[['CR',stat]].groupby(['CR']).mean().reset_index()
dfG = dfG[dfG['CR'].between(1,20)]
tfb.plot_data_and_fit(fig, 
    x=dfG['CR'],
    y=dfG[stat],
    line_color='black', 
    name='baseline',
    legendgroup='baseline',
    hovertemplate='<b>baseline</b><br>level %{x:.0f}<br>AB %{y:.1f}<extra></extra>',
)

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', automargin=True, range=[0,21], dtick=5, minor_dtick=1),
    yaxis=dict(title_text='monster attack bonus', dtick=2, minor_dtick=1),
    legend=dict(xanchor='left', x=0.00, yanchor='top', y=1.00),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
if SAVEFIGS:
    fig.update_layout(autosize=True, width=None, height=None)
    tfb.save_fig_html(fig, format='large', name=f'./fig-monster-adj-ab-vs-level-large')
    tfb.save_fig_html(fig, format='small', name=f'./fig-monster-adj-ab-vs-level-small')

In [315]:
# Fig. 7: monster chance to hit vs level for each difficulty
import plotly.graph_objects as go

def attack_hit_crit_prob(AC, AB):
    return max(0.05, min(0.95, 0.05*(21 + AB - AC)))

def pc_baseline_ac(level):
    return 14.7 + level/6

groups = [
    #{'name': 'Trivial encounters', 'difficulties': ['Trivial']},
    {'name': 'Easy encounters', 'difficulties': ['Easy']},
    {'name': 'Medium encounters', 'difficulties': ['Medium']},
    {'name': 'Hard encounters', 'difficulties': ['Hard']},
    {'name': 'Deadly encounters', 'difficulties': ['Deadly']},
    #{'name': 'Very Deadly encounters', 'difficulties': ['Very Deadly']},
]

fig = go.Figure()

colors = iter(COLOR_LIST)
for group in groups:
    dft = dfem[dfem['2014 difficulty'].isin(group['difficulties'])]
    dft['hit probability'] = 0.0
    for i in dft.index:
        dft.loc[i, 'hit probability'] = attack_hit_crit_prob(pc_baseline_ac(dft['party_level'][i]), dft['adj AB'][i])
    dft = dft[['party_level','hit probability']].groupby(['party_level']).mean().reset_index()

    levels = dft['party_level']
    values = dft['hit probability']
    tfb.plot_data_and_fit(fig, 
        x=levels,
        y=values,
        line_color=next(colors), 
        name=group['name'],
        legendgroup=group['name'],
        hovertemplate='<b>' + group['name'] + '</b><br>level %{x:.0f}<br>to hit %{y:.1%}<extra></extra>',
    )
    print(group['name'] + f' - {np.mean(values):.1%}')

dft = dfm[dfm['CR'].between(1,20)]
dft['hit probability'] = 0.0
for i in dft.index:
    dft.loc[i, 'hit probability'] = attack_hit_crit_prob(pc_baseline_ac(dft['CR'][i]), dft['adj AB'][i])
dft = dft[['CR','hit probability']].groupby(['CR']).mean().reset_index()
levels = dft['CR']
values = dft['hit probability']
tfb.plot_data_and_fit(fig, 
    x=levels,
    y=values,
    line_color='black', 
    name=f'baseline',
    legendgroup=f'baseline',
    hovertemplate='<b>baseline</b><br>level %{x:.0f}<br>to hit %{y:.1%}<extra></extra>',
)
print(f'baseline - {np.mean(values):.1%}')

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', automargin=True, range=[0,21], dtick=5, minor_dtick=1),
    yaxis=dict(title_text='monster chance to hit [%]', range=[0,1], dtick=0.2, minor_dtick=0.1, tickformat='.0%'),
    legend=dict(xanchor='left', x=0.00, yanchor='bottom', y=0.00),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
if SAVEFIGS:
    fig.update_layout(autosize=True, width=None, height=None)
    tfb.save_fig_html(fig, format='large', name=f'./fig-monster-hit-prob-vs-level-by-difficulty-large')
    tfb.save_fig_html(fig, format='small', name=f'./fig-monster-hit-prob-vs-level-by-difficulty-small')

Easy encounters - 49.7%
Medium encounters - 53.4%
Hard encounters - 56.3%
Deadly encounters - 56.7%
baseline - 68.5%


In [316]:
# Fig. 8: average adjusted AC by level for each difficulty
import plotly.graph_objects as go

stat = 'adj AC'

groups = [
    #{'name': 'Trivial encounters', 'difficulties': ['Trivial']},
    {'name': 'Easy encounters', 'difficulties': ['Easy']},
    {'name': 'Medium encounters', 'difficulties': ['Medium']},
    {'name': 'Hard encounters', 'difficulties': ['Hard']},
    {'name': 'Deadly encounters', 'difficulties': ['Deadly']},
    #{'name': 'Very Deadly encounters', 'difficulties': ['Very Deadly']},
]

fig = go.Figure()

colors = iter(COLOR_LIST)
for group in groups:
    dft = dfem[dfem['2014 difficulty'].isin(group['difficulties'])]
    dft = dft[['party_level',stat]].groupby(['party_level']).mean().reset_index()

    tfb.plot_data_and_fit(fig, 
        x=dft['party_level'],
        y=dft[stat],
        line_color=next(colors), 
        name=group['name'],
        legendgroup=group['name'],
        hovertemplate='<b>' + group['name'] + '</b><br>level %{x:.0f}<br>AC %{y:.1f}<extra></extra>',
    )

dfG = dfm[['CR',stat]].groupby(['CR']).mean().reset_index()
dfG = dfG[dfG['CR'].between(1,20)]
tfb.plot_data_and_fit(fig, 
    x=dfG['CR'],
    y=dfG[stat],
    line_color='black', 
    name='baseline',
    legendgroup='baseline',
    hovertemplate='<b>baseline</b><br>level %{x:.0f}<br>AC %{y:.1f}<extra></extra>',
)

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', automargin=True, range=[0,21], dtick=5, minor_dtick=1),
    yaxis=dict(title_text='monster armor class', dtick=2, minor_dtick=1),
    legend=dict(xanchor='left', x=0.00, yanchor='top', y=1.00),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
if SAVEFIGS:
    fig.update_layout(autosize=True, width=None, height=None)
    tfb.save_fig_html(fig, format='large', name=f'./fig-monster-adj-ac-vs-level-large')
    tfb.save_fig_html(fig, format='small', name=f'./fig-monster-adj-ac-vs-level-small')

In [317]:
# Fig. 9: PC chance to hit vs level for each difficulty
import plotly.graph_objects as go

def attack_hit_crit_prob(AC, AB):
    return max(0.05, min(0.95, 0.05*(21 + AB - AC)))

def pc_baseline_ab(level):
    return 4.6 + level/3

groups = [
    #{'name': 'Trivial encounters', 'difficulties': ['Trivial']},
    {'name': 'Easy encounters', 'difficulties': ['Easy']},
    {'name': 'Medium encounters', 'difficulties': ['Medium']},
    {'name': 'Hard encounters', 'difficulties': ['Hard']},
    {'name': 'Deadly encounters', 'difficulties': ['Deadly']},
    #{'name': 'Very Deadly encounters', 'difficulties': ['Very Deadly']},
]

fig = go.Figure()

colors = iter(COLOR_LIST)
for group in groups:
    dft = dfem[dfem['2014 difficulty'].isin(group['difficulties'])]
    dft['hit probability'] = 0.0
    for i in dft.index:
        dft.loc[i, 'hit probability'] = attack_hit_crit_prob(dft['adj AC'][i], pc_baseline_ab(dft['party_level'][i]))
    dft = dft[['party_level','hit probability']].groupby(['party_level']).mean().reset_index()

    levels = dft['party_level']
    values = dft['hit probability']
    tfb.plot_data_and_fit(fig, 
        x=levels,
        y=values,
        line_color=next(colors), 
        name=group['name'],
        legendgroup=group['name'],
        hovertemplate='<b>' + group['name'] + '</b><br>level %{x:.0f}<br>to hit %{y:.1%}<extra></extra>',
    )
    print(group['name'] + f' - {np.mean(values):.1%}')

dft = dfm[dfm['CR'].between(1,20)]
dft['hit probability'] = 0.0
for i in dft.index:
    dft.loc[i, 'hit probability'] = attack_hit_crit_prob(dft['adj AC'][i], pc_baseline_ab(dft['CR'][i]))
dft = dft[['CR','hit probability']].groupby(['CR']).mean().reset_index()
levels = dft['CR']
values = dft['hit probability']
tfb.plot_data_and_fit(fig, 
    x=levels,
    y=values,
    line_color='black', 
    name=f'baseline',
    legendgroup=f'baseline',
    hovertemplate='<b>baseline</b><br>level %{x:.0f}<br>to hit %{y:.1%}<extra></extra>',
)
print(f'baseline - {np.mean(values):.1%}')

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', automargin=True, range=[0,21], dtick=5, minor_dtick=1),
    yaxis=dict(title_text='PC chance to hit [%]', range=[0,1], dtick=0.2, minor_dtick=0.1, tickformat='.0%'),
    legend=dict(xanchor='left', x=0.00, yanchor='bottom', y=0.00),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
if SAVEFIGS:
    fig.update_layout(autosize=True, width=None, height=None)
    tfb.save_fig_html(fig, format='large', name=f'./fig-pc-hit-prob-vs-level-by-difficulty-large')
    tfb.save_fig_html(fig, format='small', name=f'./fig-pc-hit-prob-vs-level-by-difficulty-small')

Easy encounters - 71.5%
Medium encounters - 68.7%
Hard encounters - 68.0%
Deadly encounters - 66.1%
baseline - 60.3%


# Unused

# Monster Usage

In [ ]:
# Fig. ?: Average monster XP theory
import plotly.graph_objects as go

fig = go.Figure()

groups = [
    {'name': 'Easy', 'XP': 5*(0.30 + 0.15)/2},
    {'name': 'Medium', 'XP': 5*(0.45 + 0.30)/2},
    {'name': 'Hard', 'XP': 5*(0.70 + 0.45)/2},
    {'name': 'Deadly', 'XP': 5*(1.00 + 0.70)/2},
]

nmax_max = 10
for group in groups:
    color = COLOR_LIST[groups.index(group)]

    x = []
    y = []
    prob = lambda n: 1
    em = lambda n: dmg5e.encounter_xp_multiplier(5, n)
    for nmax in range(1,nmax_max+1):
        fn = sum([prob(n)/em(n) for n in range(1,nmax+1)])
        fd = sum([prob(n)*n for n in range(1,nmax+1)])

        fm = fn/fd
        x.append(nmax)
        y.append((group['XP']/1.2)*(fn/fd))

    fig.add_trace(go.Scatter(x=x, y=y, name=group['name'], legendgroup=group['name'], line_color=color))

    x = []
    y = []
    prob = lambda n: 0.3*(1 - 0.3)**(n - 1)
    #em = lambda n: 1
    for nmax in range(1,nmax_max+1):
        fn = sum([prob(n)/em(n) for n in range(1,nmax+1)])
        fd = sum([prob(n)*n for n in range(1,nmax+1)])

        fm = fn/fd
        x.append(nmax)
        y.append((group['XP']/1.2)*(fn/fd))

    fig.add_trace(go.Scatter(x=x, y=y, name=group['name'], legendgroup=group['name'], showlegend=False, line_color=color, line_dash='dash'))

fig.add_trace(go.Scatter(x=[1,nmax], y=2*[1], mode='lines', showlegend=False, line_color='black'))

"""fn = sum([1/1 for n in range(1,nmax+1)])
fd = sum([n for n in range(1,nmax+1)])

fm = fn/fd
print(fm)"""

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='max encounter size', range=[0.5,nmax_max+0.5], tickformat='.0f', dtick=2, minor_dtick=1),
    yaxis=dict(title_text='average monster XP', range=[0,2], tickformat='.1f', dtick=0.5, minor_dtick=0.1),
    legend=dict(xanchor='right', x=1.00, yanchor='top', y=1.00),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-monster-xp-vs-max-encounter-size-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-monster-xp-vs-max-encounter-size-small')

In [141]:
fig = go.Figure()


monsters = list(range(1,21))
multiplier = [dmg5e.encounter_xp_multiplier(4, m) for m in monsters]
fig.add_scatter(x=monsters, y=multiplier, name='2014 DMG')

multiplier = np.sqrt(monsters)
fig.add_scatter(x=monsters, y=multiplier, name='sqrt')

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='monsters', automargin=True, range=[0,21], dtick=5, minor_dtick=1),
    yaxis=dict(title_text='multiplier', range=[0,5], dtick=1, minor_dtick=0.5),
    legend=dict(xanchor='left', x=0.00, yanchor='top', y=1.00),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

In [128]:
import plotly.graph_objects as go

stat = 'CR'

groups = [
    #{'name': 'Trivial encounters', 'difficulties': ['Trivial']},
    {'name': 'Easy encounters', 'difficulties': ['Easy']},
    {'name': 'Medium encounters', 'difficulties': ['Medium']},
    {'name': 'Hard encounters', 'difficulties': ['Hard']},
    {'name': 'Deadly encounters', 'difficulties': ['Deadly','Very Deadly']},
    #{'name': 'Very Deadly encounters', 'difficulties': ['Very Deadly']},
]

fig = go.Figure()

colors = iter(COLOR_LIST)
for group in groups:
    dft = dfem[dfem['2014 difficulty'].isin(group['difficulties'])]
    dft = dft[['party_level',stat]].groupby(['party_level']).mean()

    tfb.plot_data_and_fit(fig, 
        x=dft.index,
        y=dft[stat],
        line_color=next(colors), 
        name=group['name'],
        legendgroup=group['name'],
        hovertemplate='level %{x:.0f}<br>' + stat + ' %{y:.1f}<extra></extra>',
    )


fig.add_scatter(
    x=[0,21],
    y=[0,21],
    line_color='black', 
    line_dash='dash',
    showlegend=False,
    hoverinfo='skip',
)

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', automargin=True, range=[0,21], dtick=5, minor_dtick=1),
    yaxis=dict(title_text=stat, range=[0,21], dtick=5, minor_dtick=1),
    legend=dict(xanchor='left', x=0.00, yanchor='top', y=1.00),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-monster-adj-dpr-vs-level-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-monster-adj-dpr-vs-level-small')

In [129]:
# adjusted hit points by level for each difficulty
import plotly.graph_objects as go

stat = 'adj HP'

groups = [
    #{'name': 'Trivial encounters', 'difficulties': ['Trivial']},
    {'name': 'Easy encounters', 'difficulties': ['Easy']},
    {'name': 'Medium encounters', 'difficulties': ['Medium']},
    {'name': 'Hard encounters', 'difficulties': ['Hard']},
    {'name': 'Deadly encounters', 'difficulties': ['Deadly']},
    #{'name': 'Very Deadly encounters', 'difficulties': ['Very Deadly']},
]

fig = go.Figure()

dfG = dfm[['CR',stat]].groupby(['CR']).mean()

colors = iter(COLOR_LIST)
for group in groups:
    dft = dfem[dfem['2014 difficulty'].isin(group['difficulties'])]
    dft = dft[['party_level',stat]].groupby(['party_level']).mean()

    tfb.plot_data_and_fit(fig, 
        x=dft.index,
        y=[dft[stat][i]/dfG[stat][i] for i in dft.index],
        line_color=next(colors), 
        name=group['name'],
        legendgroup=group['name'],
        hovertemplate='level %{x:.0f}<br>' + stat + ' %{y:.1f}<extra></extra>',
    )

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', automargin=True, range=[0,21], dtick=5, minor_dtick=1),
    yaxis=dict(title_text=stat, range=[0,1], dtick=0.2, minor_dtick=0.1, tickformat='.0%'),
    legend=dict(xanchor='left', x=0.00, yanchor='top', y=1.00),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-monster-adj-hp-vs-level-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-monster-adj-hp-vs-level-small')

In [130]:
# adjusted armor class by level for each difficulty
import plotly.graph_objects as go

stat = 'adj AC'

groups = [
    #{'name': 'Trivial encounters', 'difficulties': ['Trivial']},
    {'name': 'Easy encounters', 'difficulties': ['Easy']},
    {'name': 'Medium encounters', 'difficulties': ['Medium']},
    {'name': 'Hard encounters', 'difficulties': ['Hard']},
    {'name': 'Deadly encounters', 'difficulties': ['Deadly']},
    #{'name': 'Very Deadly encounters', 'difficulties': ['Very Deadly']},
]

fig = go.Figure()

dfG = dfm[['CR',stat]].groupby(['CR']).mean()

colors = iter(COLOR_LIST)
for group in groups:
    dft = dfem[dfem['2014 difficulty'].isin(group['difficulties'])]
    dft = dft[['party_level',stat]].groupby(['party_level']).mean()

    tfb.plot_data_and_fit(fig, 
        x=dft.index,
        y=[dft[stat][i] - dfG[stat][i] for i in dft.index],
        line_color=next(colors), 
        name=group['name'],
        legendgroup=group['name'],
        hovertemplate='level %{x:.0f}<br>' + stat + ' %{y:.1f}<extra></extra>',
    )

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', automargin=True, range=[0,21], dtick=5, minor_dtick=1),
    yaxis=dict(title_text=stat, range=[-10,5], dtick=2, minor_dtick=1),
    legend=dict(xanchor='left', x=0.00, yanchor='top', y=1.00),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-monster-adj-hp-vs-level-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-monster-adj-hp-vs-level-small')

In [120]:
# adjusted damage per round by level for each difficulty
import plotly.graph_objects as go

stat = 'adj DPR'

groups = [
    #{'name': 'Trivial encounters', 'difficulties': ['Trivial']},
    {'name': 'Easy encounters', 'difficulties': ['Easy']},
    {'name': 'Medium encounters', 'difficulties': ['Medium']},
    {'name': 'Hard encounters', 'difficulties': ['Hard']},
    {'name': 'Deadly encounters', 'difficulties': ['Deadly']},
    #{'name': 'Very Deadly encounters', 'difficulties': ['Very Deadly']},
]

fig = go.Figure()

dfG = dfm[['CR',stat]].groupby(['CR']).mean()

colors = iter(COLOR_LIST)
for group in groups:
    dft = dfem[dfem['2014 difficulty'].isin(group['difficulties'])]
    dft = dft[['party_level',stat]].groupby(['party_level']).mean()

    tfb.plot_data_and_fit(fig, 
        x=dft.index,
        y=[dft[stat][i]/dfG[stat][i] for i in dft.index],
        line_color=next(colors), 
        name=group['name'],
        legendgroup=group['name'],
        hovertemplate='level %{x:.0f}<br>' + stat + ' %{y:.1f}<extra></extra>',
    )

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', automargin=True, range=[0,21], dtick=5, minor_dtick=1),
    yaxis=dict(title_text=stat, range=[0,1], dtick=0.2, minor_dtick=0.1, tickformat='.0%'),
    legend=dict(xanchor='left', x=0.00, yanchor='top', y=1.00),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-monster-adj-hp-vs-level-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-monster-adj-hp-vs-level-small')

In [121]:
# adjusted attack bonus by level for each difficulty
import plotly.graph_objects as go

stat = 'adj AB'

groups = [
    #{'name': 'Trivial encounters', 'difficulties': ['Trivial']},
    {'name': 'Easy encounters', 'difficulties': ['Easy']},
    {'name': 'Medium encounters', 'difficulties': ['Medium']},
    {'name': 'Hard encounters', 'difficulties': ['Hard']},
    {'name': 'Deadly encounters', 'difficulties': ['Deadly']},
    #{'name': 'Very Deadly encounters', 'difficulties': ['Very Deadly']},
]

fig = go.Figure()

dfG = dfm[['CR',stat]].groupby(['CR']).mean()

colors = iter(COLOR_LIST)
for group in groups:
    dft = dfem[dfem['2014 difficulty'].isin(group['difficulties'])]
    dft = dft[['party_level',stat]].groupby(['party_level']).mean()

    tfb.plot_data_and_fit(fig, 
        x=dft.index,
        y=[dft[stat][i] - dfG[stat][i] for i in dft.index],
        line_color=next(colors), 
        name=group['name'],
        legendgroup=group['name'],
        hovertemplate='level %{x:.0f}<br>' + stat + ' %{y:.1f}<extra></extra>',
    )

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', automargin=True, range=[0,21], dtick=5, minor_dtick=1),
    yaxis=dict(title_text=stat, range=[-10,5], dtick=2, minor_dtick=1),
    legend=dict(xanchor='left', x=0.00, yanchor='top', y=1.00),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-monster-adj-hp-vs-level-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-monster-adj-hp-vs-level-small')

In [133]:
# encounter adjusted hit points by level for each difficulty
import plotly.graph_objects as go

stat = 'eHP'

groups = [
    #{'name': 'Trivial encounters', 'difficulties': ['Trivial']},
    {'name': 'Easy encounters', 'difficulties': ['Easy']},
    {'name': 'Medium encounters', 'difficulties': ['Medium']},
    {'name': 'Hard encounters', 'difficulties': ['Hard']},
    {'name': 'Deadly encounters', 'difficulties': ['Deadly']},
    #{'name': 'Very Deadly encounters', 'difficulties': ['Very Deadly']},
]

fig = go.Figure()

dfG = dfm[['CR',stat]].groupby(['CR']).mean()

colors = iter(COLOR_LIST)
for group in groups:
    dft = dfem[dfem['2014 difficulty'].isin(group['difficulties'])]
    dft = dft[['party_level','Encounter ID',stat]].groupby(['party_level','Encounter ID']).sum().reset_index()
    dft = dft[['party_level',stat]].groupby(['party_level']).mean()

    tfb.plot_data_and_fit(fig, 
        x=dft.index,
        y=[dft[stat][i]/dfG[stat][i] for i in dft.index],
        line_color=next(colors), 
        name=group['name'],
        legendgroup=group['name'],
        hovertemplate='level %{x:.0f}<br>' + stat + ' %{y:.1f}<extra></extra>',
    )

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', automargin=True, range=[0,21], dtick=5, minor_dtick=1),
    yaxis=dict(title_text=stat, range=[0,4], dtick=1.0, minor_dtick=0.2, tickformat='.0%'),
    legend=dict(xanchor='left', x=0.00, yanchor='top', y=1.00),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-monster-adj-hp-vs-level-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-monster-adj-hp-vs-level-small')

In [134]:
# encounter adjusted damage per round by level for each difficulty
import plotly.graph_objects as go

stat = 'eDPR'

groups = [
    #{'name': 'Trivial encounters', 'difficulties': ['Trivial']},
    {'name': 'Easy encounters', 'difficulties': ['Easy']},
    {'name': 'Medium encounters', 'difficulties': ['Medium']},
    {'name': 'Hard encounters', 'difficulties': ['Hard']},
    {'name': 'Deadly encounters', 'difficulties': ['Deadly']},
    #{'name': 'Very Deadly encounters', 'difficulties': ['Very Deadly']},
]

fig = go.Figure()

dfG = dfm[['CR',stat]].groupby(['CR']).mean()

colors = iter(COLOR_LIST)
for group in groups:
    dft = dfem[dfem['2014 difficulty'].isin(group['difficulties'])]
    dft = dft[['party_level','Encounter ID',stat]].groupby(['party_level','Encounter ID']).sum().reset_index()
    dft = dft[['party_level',stat]].groupby(['party_level']).mean()

    tfb.plot_data_and_fit(fig, 
        x=dft.index,
        y=[dft[stat][i]/dfG[stat][i] for i in dft.index],
        line_color=next(colors), 
        name=group['name'],
        legendgroup=group['name'],
        hovertemplate='level %{x:.0f}<br>' + stat + ' %{y:.1f}<extra></extra>',
    )

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', automargin=True, range=[0,21], dtick=5, minor_dtick=1),
    yaxis=dict(title_text=stat, range=[0,5], dtick=1.0, minor_dtick=0.2, tickformat='.0%'),
    legend=dict(xanchor='left', x=0.00, yanchor='top', y=1.00),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-monster-adj-hp-vs-level-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-monster-adj-hp-vs-level-small')

In [299]:
# Fig. 5: distribution of encounter size for each difficulty
import plotly.graph_objects as go

groups = [
    #{'name': 'Trivial encounters', 'difficulties': ['Trivial']},
    {'name': 'Easy encounters', 'difficulties': ['Easy']},
    {'name': 'Medium encounters', 'difficulties': ['Medium']},
    {'name': 'Hard encounters', 'difficulties': ['Hard']},
    {'name': 'Deadly encounters', 'difficulties': ['Deadly']},
    #{'name': 'Very Deadly encounters', 'difficulties': ['Very Deadly']},
]

fig = go.Figure()

for group in groups:
    name = group['name']
    df1 = dfe[dfe['2014 difficulty'].isin(group['difficulties'])]
    y, x = np.histogram(df1['n_monsters'], bins=np.linspace(1,21,21)-0.5)
    x = (x[1:] + x[0:-1])/2
    fig.add_trace(go.Scatter(
        x=x, 
        y=x*y/sum(y),
        #y=[d5e.encounter_multiplier_DMG(4, xx)*yy/sum(y) for xx, yy in zip(x,y)],
        mode='markers+lines',
        name=name,
        showlegend=True,
        hovertemplate = f'<b>{name}</b><br>'
                + 'monsters %{x:.0f}<br>'
                + 'probability %{y:.1%}'
                + '<extra></extra>'
    ))

y, x = np.histogram(dfe['n_monsters'], bins=np.linspace(1,21,21)-0.5)
x = (x[1:] + x[0:-1])/2
fig.add_trace(go.Scatter(
    x=x, 
    y=x*y/sum(y),
    mode='markers+lines',
    name='all encounters',
    line_color='black',
    line_dash='dash',
    showlegend=True,
    hovertemplate = f'<b>all encounters</b><br>'
                    + 'monsters %{x:.0f}<br>'
                    + 'probability %{y:.1%}'
                    + '<extra></extra>'
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='monsters per encounter', range=[0,10], dtick=1, minor_dtick=1),
    yaxis=dict(title_text='encounters [%]'),
    legend=dict(
        xanchor='right', x=1.00, 
        yanchor='top', y=1.00,
        orientation='v',
    ),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-monsters-per-encounter-by-difficulty-large', style='width: 600px;')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-monsters-per-encounter-by-difficulty-small', style='width: 600px;')

In [132]:
xp_mean = [
    (0.15 + 0.30)/2,
    (0.30 + 0.45)/2,
    (0.45 + 0.70)/2,
    (0.70 + 1.00)/2,
]
for i in range(4):
    print('{0:.2f}'.format(np.sqrt(xp_mean[i]/xp_mean[0])))

1.00
1.29
1.60
1.94


In [201]:
# Plots a histogram of monsters per encounter
import plotly.graph_objects as go

#difficulties = ['Easy','Medium','Hard','Deadly','Very Deadly']
difficulties = ['Medium']

fig = go.Figure()


dft = dfem[dfem['2014 difficulty'].isin(difficulties)]
#dft = dft[['party_level','XP ratio']].groupby(['party_level']).mean().reset_index()
monster_xp_ratios = dft['XP ratio']

"""
monster_xp_ratios  = []
for level in levels:
    dft = dfe[dfe['party_level'].eq(level) & dfe['2014 difficulty'].isin(difficulties)]
    monster_xp_values = []
    for x in dft['monsters_xp']:
        monster_xp_values.extend(x)

    pc_xp_budget = 0.5*player_character_xp_budget(level)

    monster_xp_ratios.extend([xp/pc_xp_budget for xp in monster_xp_values])
"""

y, x = np.histogram(monster_xp_ratios, bins=np.linspace(0,3,31))
x = (x[1:] + x[0:-1])/2
fig.add_trace(go.Scatter(
    x=x, 
    #y=y/len(monster_xp_ratios),
    y=x*np.sqrt(x)*y/np.trapezoid(y, x*np.sqrt(x)),
    #y=[d5e.encounter_multiplier_DMG(4, xx)*yy/sum(y) for xx, yy in zip(x,y)],
    mode='markers+lines',
    #name=name,
    #showlegend=False,
    #hovertemplate = f'<b>{name}</b><br>'
    #        + 'monsters %{x:.0f}<br>'
    #        + 'probability %{y:.1%}'
    #        + '<extra></extra>'
))

"""fig.add_trace(go.Histogram(
    x=monster_xp_ratios, 
    xbins_size=0.05, 
    histnorm='probability', 
    showlegend=False,
    #hovertemplate = ''
    #                + 'monsters %{x:.0f}<br>'
    #                + 'probability %{y:.1%}'
    #                + '<extra></extra>'
))"""

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    #barmode='stack',
    xaxis=dict(title_text='XP ratio', range=[0,3], tickformat='.2f', dtick=0.2, minor_dtick=0.1),
    yaxis=dict(title_text='monsters (%)', tickformat='.0%'),
    width=600, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-monsters-per-encounter-large', style='width: 600px;')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-monsters-per-encounter-small', style='width: 600px;')

In [274]:
# Plots a histogram of monsters per encounter
import plotly.graph_objects as go
import numpy as np
import plotly.colors as pc

difficulties = ['Easy','Medium','Hard','Deadly','Very Deadly']
difficulties = ['Medium']

fig = go.Figure()

y_total = 0
norm = dfe[dfe['2014 difficulty'].isin(difficulties)]['n_monsters'].sum()
nmax = 5
levels = list(range(1,21))
for n in range(1,nmax+1):
    monster_xp_ratios  = []
    for level in levels:
        dft = dfe[dfe['party_level'].eq(level) & dfe['2014 difficulty'].isin(difficulties) & dfe['n_monsters'].eq(n) & dfe['n_unique_monsters'].eq(1)]
        monster_xp_values = []
        for x in dft['monsters_xp']:
            monster_xp_values.extend(x)

        pc_xp_budget = 0.5*player_character_xp_budget(level)

        monster_xp_ratios.extend([xp/pc_xp_budget for xp in monster_xp_values])

    y, x = np.histogram(monster_xp_ratios, bins=80, range=[0,4], density=None)
    y_total += y
    fig.add_trace(go.Bar(
        x=x,
        y=y/norm, 
        name=f'{n} monsters',
        legendgroup=f'{n} monsters',
        marker_line_width=0,
    ))

ymax = 1.1*max(y_total)/norm
"""
colors = iter(COLOR_LIST)
for n in range(1,nmax+1):
    m = dmg5e.encounter_xp_multiplier(5, n)
    #xmin, xmax = 0.42*5/(n*m), 0.74*5/(n*m)
    xmin, xmax = 0.28*5/(n*m), 0.47*5/(n*m)
    color = next(colors)
    fig.add_trace(go.Scatter(
        x=[xmin, xmax], 
        y=[ymax, ymax],
        mode='none',
        fill='tozeroy', 
        showlegend=False,
        legendgroup=f'{n} monsters',
        fillcolor='rgba({0},{1},{2},0.25)'.format(*pc.hex_to_rgb(color)),
        hoverinfo='none'
    ))
    #fig.add_annotation(
    #    x=(xmin + xmax)/2, 
    #    y=0.95*ymax,
    #    text=f'{n}',
    #    showarrow=False,
    #    font_size=12,
    #)
"""

"""
fig.add_trace(go.Histogram(
    x=monster_xp_ratios, 
    xbins_size=0.05, 
    histnorm='probability', 
    showlegend=False,
    #hovertemplate = ''
    #                + 'monsters %{x:.0f}<br>'
    #                + 'probability %{y:.1%}'
    #                + '<extra></extra>'
))"""

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    barmode='stack',
    bargap=0,
    xaxis=dict(title_text='XP ratio', range=[0,4], tickformat='.2f', dtick=1.0, minor_dtick=0.1),
    yaxis=dict(title_text='monsters (%)', range=[0,ymax], tickformat='.0%'),
    legend=dict(xanchor='right', x=1.00, yanchor='top', y=1.00),
    width=600, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-monsters-per-encounter-large', style='width: 600px;')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-monsters-per-encounter-small', style='width: 600px;')

In [ ]:
# Plots a histogram of monsters per encounter
import plotly.graph_objects as go
import numpy as np
import plotly.colors as pc

difficulties = ['Easy','Medium','Hard','Deadly','Very Deadly']
difficulties = ['Medium']

fig = go.Figure()

y_total = 0
norm = dfe[dfe['2014 difficulty'].isin(difficulties)]['n_monsters'].sum()
nmax = 5
levels = list(range(1,21))
for n in range(1,nmax+1):
    monster_xp_ratios  = []
    for level in levels:
        dft = dfe[dfe['party_level'].eq(level) & dfe['2014 difficulty'].isin(difficulties) & dfe['n_monsters'].eq(n) & dfe['n_unique_monsters'].eq(1)]
        monster_xp_values = []
        for x in dft['monsters_xp']:
            monster_xp_values.extend(x)

        pc_xp_budget = 0.5*player_character_xp_budget(level)

        monster_xp_ratios.extend([xp/pc_xp_budget for xp in monster_xp_values])

    y, x = np.histogram(monster_xp_ratios, bins=40, range=[0,4], density=None)
    y_total += y
    fig.add_trace(go.Bar(
        x=x,
        y=y/norm, 
        name=f'{n} monsters',
        legendgroup=f'{n} monsters',
        marker_line_width=0,
    ))

ymax = 1.1*max(y_total)/norm
"""
colors = iter(COLOR_LIST)
for n in range(1,nmax+1):
    m = dmg5e.encounter_xp_multiplier(5, n)
    #xmin, xmax = 0.42*5/(n*m), 0.74*5/(n*m)
    xmin, xmax = 0.28*5/(n*m), 0.47*5/(n*m)
    color = next(colors)
    fig.add_trace(go.Scatter(
        x=[xmin, xmax], 
        y=[ymax, ymax],
        mode='none',
        fill='tozeroy', 
        showlegend=False,
        legendgroup=f'{n} monsters',
        fillcolor='rgba({0},{1},{2},0.25)'.format(*pc.hex_to_rgb(color)),
        hoverinfo='none'
    ))
    #fig.add_annotation(
    #    x=(xmin + xmax)/2, 
    #    y=0.95*ymax,
    #    text=f'{n}',
    #    showarrow=False,
    #    font_size=12,
    #)
"""

"""
fig.add_trace(go.Histogram(
    x=monster_xp_ratios, 
    xbins_size=0.05, 
    histnorm='probability', 
    showlegend=False,
    #hovertemplate = ''
    #                + 'monsters %{x:.0f}<br>'
    #                + 'probability %{y:.1%}'
    #                + '<extra></extra>'
))"""

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    bargap=0,
    xaxis=dict(title_text='XP ratio', range=[0,4], tickformat='.2f', dtick=1.0, minor_dtick=0.1),
    yaxis=dict(title_text='monsters (%)', range=[0,ymax], tickformat='.0%'),
    legend=dict(xanchor='right', x=1.00, yanchor='top', y=1.00),
    width=600, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-monsters-per-encounter-large', style='width: 600px;')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-monsters-per-encounter-small', style='width: 600px;')

In [252]:
# Plots a histogram of monsters per encounter
import plotly.graph_objects as go
import numpy as np
import plotly.colors as pc

groups = [
    #{'name': 'Trivial', 'difficulties': ['Trivial']},
    {'name': 'Easy', 'difficulties': ['Easy'], 'n_monsters': [1,2,3,4], 'xp_range': (0.15,0.30)},
    {'name': 'Medium', 'difficulties': ['Medium'], 'n_monsters': [1,2,3,4], 'xp_range': (0.30,0.45)},
    {'name': 'Hard', 'difficulties': ['Hard'], 'n_monsters': [1,2,3,4], 'xp_range': (0.45,0.70)},
    {'name': 'Deadly', 'difficulties': ['Deadly'], 'n_monsters': [1,2,3,4], 'xp_range': (0.70,1.00)},
    #{'name': 'Very Deadly', 'difficulties': ['Very Deadly']},
]

fig = go.Figure()

for group in groups:
    name = group['name']
    monster_xp_ratios  = []
    for level in levels:
        dft = dfe[dfe['party_level'].eq(level) 
            & dfe['2014 difficulty'].isin(group['difficulties']) 
            & dfe['n_monsters'].isin(group['n_monsters']) 
            & dfe['n_unique_monsters'].eq(1)]
        monster_xp_values = []
        for x in dft['monsters_xp']:
            monster_xp_values.extend(x)

        pc_xp_budget = 0.5*player_character_xp_budget(level)

        monster_xp_ratios.extend([xp/pc_xp_budget for xp in monster_xp_values])
    
    #y, x = np.histogram(monster_xp_ratios, bins=20, range=[0,8*group['x_scale']], density=None)
    xp_mid = np.mean(group['xp_range'])
    xp_min, xp_max = group['xp_range']
    y, x = np.histogram(monster_xp_ratios, bins=31, range=[0,6*xp_max], density=None)
    x = 0.5*(x[0:-1] + x[1:])
    y = y/sum(y)
    x = x/(5*xp_mid)
    fig.add_trace(go.Scatter(
        x=x,
        y=y, 
        name=name,
        legendgroup=name,
        fill='tozeroy',
        #marker_line_width=0,
    ))


# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    barmode='stack',
    bargap=0,
    xaxis=dict(title_text='XP ratio', range=[0,1.5], tickformat='.2f', dtick=0.5, minor_dtick=0.1),
    yaxis=dict(title_text='monsters (%)', range=[0,0.4], tickformat='.0%'),
    legend=dict(xanchor='right', x=1.00, yanchor='top', y=1.00),
    width=600, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-monsters-per-encounter-large', style='width: 600px;')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-monsters-per-encounter-small', style='width: 600px;')

In [296]:
# Plots a histogram of monsters per encounter
import plotly.graph_objects as go
import numpy as np
import plotly.colors as pc
from plotly.subplots import make_subplots

n_monsters = list(range(1,6))
groups = [
    #{'name': 'Trivial', 'difficulties': ['Trivial']},
    {'name': 'Easy', 'difficulties': ['Easy'], 'n_monsters': n_monsters, 'xp_range': (0.15,0.30)},
    {'name': 'Medium', 'difficulties': ['Medium'], 'n_monsters': n_monsters, 'xp_range': (0.30,0.45)},
    {'name': 'Hard', 'difficulties': ['Hard'], 'n_monsters': n_monsters, 'xp_range': (0.45,0.70)},
    {'name': 'Deadly', 'difficulties': ['Deadly'], 'n_monsters': n_monsters, 'xp_range': (0.70,1.00)},
    #{'name': 'Very Deadly', 'difficulties': ['Very Deadly']},
]

fig = make_subplots(
    rows=len(groups), 
    cols=1, 
    shared_xaxes=True, 
    vertical_spacing=0.02, 
    subplot_titles=[g['name'] for g in groups],
)


for group in groups:
    row = 1 + groups.index(group)
    name = group['name']
    monster_xp_ratios  = []
    for level in levels:
        dft = dfe[dfe['party_level'].eq(level) 
            & dfe['2014 difficulty'].isin(group['difficulties']) 
            & dfe['n_monsters'].isin(group['n_monsters']) 
            & dfe['n_unique_monsters'].eq(1)
        ]
        monster_xp_values = []
        for x in dft['monsters_xp']:
            monster_xp_values.extend(x)

        pc_xp_budget = 0.5*player_character_xp_budget(level)

        monster_xp_ratios.extend([xp/pc_xp_budget for xp in monster_xp_values])
    
    #y, x = np.histogram(monster_xp_ratios, bins=20, range=[0,8*group['x_scale']], density=None)
    xp_mid = np.mean(group['xp_range'])
    xp_min, xp_max = group['xp_range']
    y, x = np.histogram(monster_xp_ratios, bins=31, range=[0,6*xp_max], density=None)
    x = 0.5*(x[0:-1] + x[1:])
    x = np.concat(([0], x))
    y = np.concat(([0], y))
    y = y/sum(y)
    #x = x/(5*xp_mid)
    fig.add_trace(go.Scatter(
        x=x,
        y=y, 
        name=name,
        legendgroup=name,
        fill='tozeroy',
        showlegend=False,
        #marker_line_width=0,
    ), row=row, col=1)


# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    barmode='stack',
    bargap=0,
    xaxis4=dict(title_text='XP ratio', range=[0,5], tickformat='.2f', dtick=1.0, minor_dtick=0.2, automargin=True),
    yaxis=dict(title_text='monsters (%)', range=[0,0.35], tickformat='.0%', dtick=0.1, minor_dtick=0.05),
    yaxis2=dict(title_text='monsters (%)', range=[0,0.35], tickformat='.0%', dtick=0.1, minor_dtick=0.05),
    yaxis3=dict(title_text='monsters (%)', range=[0,0.35], tickformat='.0%', dtick=0.1, minor_dtick=0.05),
    yaxis4=dict(title_text='monsters (%)', range=[0,0.35], tickformat='.0%', dtick=0.1, minor_dtick=0.05),
    legend=dict(xanchor='right', x=1.00, yanchor='top', y=1.00),
    width=550, 
    height=600,
)
for a in fig.layout.annotations:
    a.update(y=a['y']-0.05)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-monsters-per-encounter-large', style='width: 600px;')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-monsters-per-encounter-small', style='width: 600px;')

In [286]:
# Plots a histogram of monsters per encounter
import plotly.graph_objects as go
import numpy as np

groups = [
    #{'name': 'different', 'difficulties': ['Medium'], 'n_monsters': [2], 'n_unique_monsters': [2]},
    #{'name': 'same', 'difficulties': ['Medium'], 'n_monsters': [2], 'n_unique_monsters': [1]},

    {'name': '3 types', 'difficulties': ['Deadly'], 'n_monsters': [3], 'n_unique_monsters': [3]},
    {'name': '2 types', 'difficulties': ['Deadly'], 'n_monsters': [3], 'n_unique_monsters': [2]},
    {'name': '1 type', 'difficulties': ['Deadly'], 'n_monsters': [3], 'n_unique_monsters': [1]},
]

fig = go.Figure()

norm = dfe[dfe['2014 difficulty'].isin(difficulties)]['n_monsters'].sum()
levels = list(range(1,21))
for group in groups:
    monster_xp_ratios  = []
    for level in levels:
        dft = dfe[
            dfe['party_level'].eq(level) & 
            dfe['2014 difficulty'].isin(group['difficulties']) & 
            dfe['n_monsters'].isin(group['n_monsters']) & 
            dfe['n_unique_monsters'].isin(group['n_unique_monsters'])
        ]
        monster_xp_values = []
        for x in dft['monsters_xp']:
            monster_xp_values.extend(x)

        pc_xp_budget = 0.5*player_character_xp_budget(level)

        monster_xp_ratios.extend([xp/pc_xp_budget for xp in monster_xp_values])

    y, x = np.histogram(monster_xp_ratios, bins=40, range=[0,4], density=None)
    fig.add_trace(go.Bar(
        x=x,
        y=y/norm, 
        name=group['name'],
        legendgroup=group['name'],
        marker_line_width=0,
    ))

"""
colors = iter(COLOR_LIST)
for n in range(1,nmax+1):
    m = dmg5e.encounter_xp_multiplier(5, n)
    #xmin, xmax = 0.42*5/(n*m), 0.74*5/(n*m)
    xmin, xmax = 0.28*5/(n*m), 0.47*5/(n*m)
    color = next(colors)
    fig.add_trace(go.Scatter(
        x=[xmin, xmax], 
        y=[ymax, ymax],
        mode='none',
        fill='tozeroy', 
        showlegend=False,
        legendgroup=f'{n} monsters',
        fillcolor='rgba({0},{1},{2},0.25)'.format(*pc.hex_to_rgb(color)),
        hoverinfo='none'
    ))
    #fig.add_annotation(
    #    x=(xmin + xmax)/2, 
    #    y=0.95*ymax,
    #    text=f'{n}',
    #    showarrow=False,
    #    font_size=12,
    #)
"""

"""
fig.add_trace(go.Histogram(
    x=monster_xp_ratios, 
    xbins_size=0.05, 
    histnorm='probability', 
    showlegend=False,
    #hovertemplate = ''
    #                + 'monsters %{x:.0f}<br>'
    #                + 'probability %{y:.1%}'
    #                + '<extra></extra>'
))"""

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    barmode='stack',
    bargap=0,
    xaxis=dict(title_text='XP ratio', range=[0,4], tickformat='.2f', dtick=0.5, minor_dtick=0.1),
    yaxis=dict(title_text='monsters (%)', tickformat='.1%'),
    legend=dict(xanchor='right', x=1.00, yanchor='top', y=1.00),
    width=600, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-monsters-per-encounter-large', style='width: 600px;')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-monsters-per-encounter-small', style='width: 600px;')

In [240]:
# Plots a histogram of monsters per encounter
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=dfm[dfm['book appearances'].gt(0)]['book appearances'], 
    xbins_size=1, 
    histnorm='probability', 
    showlegend=False,
    hovertemplate = ''
                    + 'monsters %{x:.0f}<br>'
                    + 'probability %{y:.1%}'
                    + '<extra></extra>'
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    #barmode='stack',
    xaxis=dict(title_text='appearances', range=[0,100], tickformat='.0f', dtick=5, minor_dtick=1),
    yaxis=dict(title_text='monsters (%)'),
    width=600, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-monsters-per-encounter-large', style='width: 600px;')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-monsters-per-encounter-small', style='width: 600px;')

In [243]:
# Plots a histogram of monsters per encounter
import plotly.graph_objects as go

fig = go.Figure()

dft = dfm.nlargest(10, 'book appearances')

fig.add_trace(go.Bar(
    x=dft['Monster'],
    y=dft['book appearances'], 
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    #barmode='stack',
    #xaxis=dict(title_text='appearances', range=[0,100], tickformat='.0f', dtick=5, minor_dtick=1),
    xaxis=dict(automargin=True),
    yaxis=dict(title_text='appearances', dtick=50, minor_dtick=10, automargin=True),
    width=600, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-monsters-per-encounter-large', style='width: 600px;')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-monsters-per-encounter-small', style='width: 600px;')

In [244]:
# Plots a histogram of monsters per encounter
import plotly.graph_objects as go

fig = go.Figure()

dfG0 = dfm[['Book','Monster ID']].groupby('Book').count()

dft = dfm[dfm['book appearances'].gt(0)]
dfG1 = dft[['Book','Monster ID']].groupby('Book').count()

#books = dfG1.index.to_list()
books = ['MM (2014)','VGtM','MToF','VRGtR','MPMotM']
fig.add_trace(go.Bar(
    x=[book for book in books],
    y=[dfG1['Monster ID'][book]/dfG0['Monster ID'][book] for book in books], 
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    #barmode='stack',
    #xaxis=dict(title_text='appearances', range=[0,100], tickformat='.0f', dtick=5, minor_dtick=1),
    xaxis=dict(automargin=True),
    yaxis=dict(title_text='monsters used [%]', range=[0,1], dtick=0.2, minor_dtick=0.1, automargin=True, tickformat='.0%'),
    width=600, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-monsters-per-encounter-large', style='width: 600px;')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-monsters-per-encounter-small', style='width: 600px;')

In [245]:
# Plots a histogram of monsters per encounter
import plotly.graph_objects as go

fig = go.Figure()

dfG0 = dfm[['Book','Monster ID']].groupby('Book').count()

dft = dfm[dfm['book appearances'].gt(0)]
dfG1 = dft[['Book','Monster ID']].groupby('Book').count()

books = dfG1.index.to_list()
exclude_books = ['MM (2014)','VGtM','MToF','VRGtR','MPMotM','OotA','TWBtW','SAiS:BAM','MM (2024)']
books = [book for book in books if book not in exclude_books]
fig.add_trace(go.Bar(
    x=[book for book in books],
    y=[dfG1['Monster ID'][book]/dfG0['Monster ID'][book] for book in books], 
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    #barmode='stack',
    #xaxis=dict(title_text='appearances', range=[0,100], tickformat='.0f', dtick=5, minor_dtick=1),
    xaxis=dict(automargin=True),
    yaxis=dict(title_text='monsters used [%]', range=[0,1], dtick=0.2, minor_dtick=0.1, automargin=True, tickformat='.0%'),
    width=600, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-monsters-per-encounter-large', style='width: 600px;')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-monsters-per-encounter-small', style='width: 600px;')

## Monster XP ratios

In [56]:
# Plots a histogram of encounter XPs by book
import plotly.graph_objects as go

groups = [
    #{'name': 'Trivial encounters', 'difficulties': ['Trivial']},
    {'name': 'Easy encounters', 'difficulties': ['Easy']},
    {'name': 'Medium encounters', 'difficulties': ['Medium']},
    {'name': 'Hard encounters', 'difficulties': ['Hard']},
    {'name': 'Deadly encounters', 'difficulties': ['Deadly']},
    #{'name': 'Very Deadly encounters', 'difficulties': ['Very Deadly']},
]

ymax = 0.35

fig = go.Figure()

for group in groups:
    monster_xp_ratios  = []
    name = group['name']
    difficulties = group['difficulties']
    for level in levels:
        dft = dfe[dfe['party_level'].eq(level) & dfe['2014 difficulty'].isin(difficulties)]
        monster_xp_values = []
        for x in dft['monsters_xp']:
            monster_xp_values.extend(x)

        pc_xp_budget = 0.5*player_character_xp_budget(level)

        monster_xp_ratios.extend([xp/pc_xp_budget for xp in monster_xp_values])

    nbins = 20
    y, x = np.histogram(monster_xp_ratios, bins=np.linspace(0,2,nbins+1))
    x = (x[1:] + x[0:-1])/2
    x = [x[i] for i in range(nbins) if y[i] > 0]
    y = [y[i] for i in range(nbins) if y[i] > 0]
    fig.add_trace(go.Scatter(
        x=x, 
        y=y/sum(y),
        #y=[d5e.encounter_multiplier_DMG(4, xx)*yy/sum(y) for xx, yy in zip(x,y)],
        mode='markers+lines',
        name=name,
        hovertemplate = f'<b>{name}</b><br>'
                + 'XP ratio %{x:.2f}<br>'
                + 'encounters %{y:.1%}'
                + '<extra></extra>'
    ))

"""y, x = np.histogram(df['n_monsters'], bins=np.linspace(1,21,21)-0.5)
x = (x[1:] + x[0:-1])/2
fig.add_trace(go.Scatter(
    x=x, 
    y=y/sum(y),
    #y=[d5e.encounter_multiplier_DMG(4, xx)*yy/sum(y) for xx, yy in zip(x,y)],
    mode='markers+lines',
    name='average',
    line_color='black',
    line_dash='dash',
    showlegend=True,
    hovertemplate = f'<b>Average</b><br>'
                    + 'monsters %{x:.0f}<br>'
                    + 'probability %{y:.1%}'
                    + '<extra></extra>'
))"""

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='monster XP ratio', range=[0,2], dtick=0.5, minor_dtick=0.1),
    yaxis=dict(title_text='encounters [%]', range=[-0.01, 0.51], tickformat='.0%'),
    legend=dict(
        xanchor='right', x=1.00, 
        yanchor='top', y=1.00,
        orientation='v',
    ),
    width=600, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-monsters-per-encounter-by-difficulty-large', style='width: 600px;')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-monsters-per-encounter-by-difficulty-small', style='width: 600px;')

In [54]:
# Plots a histogram of encounter XPs by book
import plotly.graph_objects as go

groups = [
    #{'name': 'Trivial encounters', 'difficulties': ['Trivial']},
    #{'name': 'Easy encounters', 'difficulties': ['Easy']},
    #{'name': 'Medium encounters', 'difficulties': ['Medium']},
    #{'name': 'Hard encounters', 'difficulties': ['Hard']},
    #{'name': 'Deadly encounters', 'difficulties': ['Deadly']},
    #{'name': 'Very Deadly encounters', 'difficulties': ['Very Deadly']},
    {'name': 'all encounters', 'difficulties': ['Deadly','Very Deadly']},
]

ymax = 0.35

fig = go.Figure()

for group in groups:
    monster_xp_ratios  = []
    name = group['name']
    difficulties = group['difficulties']
    for level in levels:
        dft = dfe[dfe['party_level'].eq(level) & dfe['2014 difficulty'].isin(difficulties)]
        monster_xp_values = []
        for x in dft['monsters_xp']:
            monster_xp_values.extend(x)

        pc_xp_budget = 0.5*player_character_xp_budget(level)

        monster_xp_ratios.extend([xp/pc_xp_budget for xp in monster_xp_values])

    nbins = 10
    y, x = np.histogram(monster_xp_ratios, bins=np.linspace(0,2,nbins+1))
    x = (x[1:] + x[0:-1])/2
    x = [x[i] for i in range(nbins) if y[i] > 0]
    y = [y[i] for i in range(nbins) if y[i] > 0]
    x = np.array(x)
    y = np.array(y)
    fig.add_trace(go.Scatter(
        x=x, 
        y=x*y/np.trapezoid(y*x), 
        #y=[d5e.encounter_multiplier_DMG(4, xx)*yy/sum(y) for xx, yy in zip(x,y)],
        mode='markers+lines',
        name=name,
        hovertemplate = f'<b>{name}</b><br>'
                + 'XP ratio %{x:.2f}<br>'
                + 'encounters %{y:.1%}'
                + '<extra></extra>'
    ))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    #xaxis=dict(title_text='monster XP ratio', range=[0,2], dtick=0.5, minor_dtick=0.1),
    #yaxis=dict(title_text='encounters [%]', range=[-0.01, 0.51], tickformat='.0%'),
    xaxis=dict(title_text='monster XP ratio'),
    yaxis=dict(title_text='encounters [%]'),
    legend=dict(
        xanchor='right', x=1.00, 
        yanchor='top', y=1.00,
        orientation='v',
    ),
    width=600, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-monsters-per-encounter-by-difficulty-large', style='width: 600px;')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-monsters-per-encounter-by-difficulty-small', style='width: 600px;')

## Monsters per Encounter

In [246]:
# Plots a histogram of monsters per encounter
import plotly.graph_objects as go

fig = go.Figure()
ymax = 0.35
"""xbins=dict(
        start=-3.0,
        end=4,
        size=0.5
    )"""
fig.add_trace(go.Histogram(
    x=dfe['n_monsters'], 
    xbins_size=1, 
    histnorm='probability', 
    showlegend=False,
    hovertemplate = ''
                    + 'monsters %{x:.0f}<br>'
                    + 'probability %{y:.1%}'
                    + '<extra></extra>'
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    #barmode='stack',
    xaxis=dict(title_text='monsters per encounter', range=[0,20], tickformat='.0f', dtick=5, minor_dtick=1),
    yaxis=dict(title_text='encounter (%)', range=[0,ymax], tickformat='.0%', dtick=0.05, minor_dtick=0.01),
    width=600, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-monsters-per-encounter-large', style='width: 600px;')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-monsters-per-encounter-small', style='width: 600px;')

In [247]:
# Plots a histogram of monsters per encounter
import plotly.graph_objects as go

fig = go.Figure()
ymax = 0.75
"""xbins=dict(
        start=-3.0,
        end=4,
        size=0.5
    )"""
fig.add_trace(go.Histogram(x=dfe['n_unique_monsters'], xbins_size=1, histnorm='probability', showlegend=False))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    #barmode='stack',
    xaxis=dict(title_text='unique monsters per encounter', range=[0,7], tickformat='.0f', dtick=1, minor_dtick=1),
    yaxis=dict(title_text='encounter (%)', range=[0,ymax], tickformat='.0%', dtick=0.10, minor_dtick=0.02),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')

In [248]:
# histogram of difficulties by book
import plotly.graph_objects as go
from plotly.express.colors import sample_colorscale

#books = ['ToD','PotA','SKT','ToA','W:DotMM','BG:DiA','ID:RotF','CR:CotN','D:SotDQ','PaB:TSO','V:EoR']
#books = dfe['book_acronym'].unique()
books = [v['acronym'] for k, v in book_dict.items()]
fig = go.Figure()

unique_monsters = []
for book in books:
    dft = dfe[dfe['book_acronym'].eq(book)]
    monsters = []
    for x in dft['monsters']:
        monsters.extend(x)
    unique_monsters.append(len(np.unique(monsters)))

fig.add_trace(go.Bar(
    x=books,
    y=unique_monsters, 
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    barmode='stack',
    xaxis=dict(title_text='book', automargin=True),
    yaxis=dict(title_text='unique monsters', automargin=True, tickformat='.0f', dtick=50, minor_dtick=10),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')

In [19]:
# Plots the average number of monsters per encounter by level
import plotly.graph_objects as go

levels = list(range(1,21))

groups = [
    #{'name': 'Trivial encounters', 'difficulties': ['Trivial']},
    {'name': 'Easy encounters', 'difficulties': ['Easy']},
    {'name': 'Medium encounters', 'difficulties': ['Medium']},
    {'name': 'Hard encounters', 'difficulties': ['Hard']},
    {'name': 'Deadly encounters', 'difficulties': ['Deadly']},
    #{'name': 'Very Deadly encounters', 'difficulties': ['Very Deadly']},
]

fig = go.Figure()

colors = iter(COLOR_LIST)
for group in groups:
    dft = dfe[dfe['2014 difficulty'].isin(group['difficulties'])]
    dft = dft[['party_level','n_monsters']].groupby(['party_level']).mean().reset_index()

    #fig.add_trace(go.Scatter(
    #    x=dft['party_level'],
    #    y=dft['n_monsters'],
    #    mode='markers+lines',
    #    name=group['name'],
    #    legendgroup=group['name'],
    #    #customdata=dfC.index,
    #    #hovertemplate = '<b>' + g + '</b><br>'
    #    #    + 'CR %{customdata}<br>'
    #    #    + 'Monsters %{y:,.0f}'
    #    #    + '<extra></extra>'
    #))
    tfb.plot_data_and_fit(fig, 
        x=dft['party_level'],
        y=dft['n_monsters'],
        line_color=next(colors), 
        name=group['name'],
        legendgroup=group['name'],
    )

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', range=[0,21], tickformat='.0f', dtick=5, minor_dtick=1),
    yaxis=dict(title_text='monsters per encounter', range=[0,10], tickformat='.0f', dtick=2, minor_dtick=1),
    legend=dict(xanchor='left', x=0.00, yanchor='top', y=1.00),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')

In [249]:
# Plots the average number of monsters per encounter by level
import plotly.graph_objects as go

levels = list(range(1,21))

fig = go.Figure()

dft = dfe[dfe['2014 difficulty'].isin(['Medium','Hard','Deadly','Very Deadly'])]
dft = dft[['party_level','n_monsters']].groupby(['party_level']).mean().reset_index()


fig.add_trace(go.Scatter(
    x=dft['party_level'],
    y=dft['n_monsters'],
    showlegend=False,
    mode='markers',
    #customdata=dfC.index,
    #hovertemplate = '<b>' + g + '</b><br>'
    #    + 'CR %{customdata}<br>'
    #    + 'Monsters %{y:,.0f}'
    #    + '<extra></extra>'
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', range=[0,21], tickformat='.0f', dtick=5, minor_dtick=1),
    yaxis=dict(title_text='monsters per encounter', range=[0,21], tickformat='.0f', dtick=5, minor_dtick=1),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')

In [250]:
# Plots a histogram of monsters per encounter
import plotly.graph_objects as go
from plotly.subplots import make_subplots

levels = list(range(1,21))

fig = go.Figure()

dft = dfe[dfe['2014 difficulty'].isin(['Easy','Medium','Hard','Deadly','Very Deadly'])]

dfG = dft[['party_level','n_monsters']].groupby(['party_level']).quantile(0.2).reset_index()
fig.add_trace(go.Scatter(
    x=dfG['party_level'],
    y=dfG['n_monsters'],
    showlegend=False,
    mode='lines',
    line=dict(color=COLOR_LIST[0], width=0),
    #customdata=dfC.index,
    #hovertemplate = '<b>' + g + '</b><br>'
    #    + 'CR %{customdata}<br>'
    #    + 'Monsters %{y:,.0f}'
    #    + '<extra></extra>'
))

dfG = dft[['party_level','n_monsters']].groupby(['party_level']).quantile(0.8).reset_index()
fig.add_trace(go.Scatter(
    x=dfG['party_level'],
    y=dfG['n_monsters'],
    showlegend=False,
    mode='lines',
    fill='tonexty',
    line=dict(color=COLOR_LIST[0], width=0),
    #customdata=dfC.index,
    #hovertemplate = '<b>' + g + '</b><br>'
    #    + 'CR %{customdata}<br>'
    #    + 'Monsters %{y:,.0f}'
    #    + '<extra></extra>'
))

dfG = dft[['party_level','n_monsters']].groupby(['party_level']).median().reset_index()
fig.add_trace(go.Scatter(
    x=dfG['party_level'],
    y=dfG['n_monsters'],
    showlegend=False,
    mode='lines',
    line=dict(color=COLOR_LIST[0], width=2),
    #customdata=dfC.index,
    #hovertemplate = '<b>' + g + '</b><br>'
    #    + 'CR %{customdata}<br>'
    #    + 'Monsters %{y:,.0f}'
    #    + '<extra></extra>'
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', range=[0,21], tickformat='.0f', dtick=5, minor_dtick=1),
    yaxis=dict(title_text='monsters per encounter', range=[0,10], tickformat='.0f', dtick=2, minor_dtick=1),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')

## Median monster XP by level

In [252]:
# Plots a histogram of monsters per encounter
import plotly.graph_objects as go
from plotly.subplots import make_subplots

levels = list(range(1,18))

#fig = go.Figure()
fig = make_subplots(rows=len(levels), cols=1, shared_xaxes=True, vertical_spacing=0.005, subplot_titles=[f'level {x}' for x in levels])


def x_decoder(x):
    match x:
        case 0:
            return 0
        case 0.125:
            return 0.125
        case 0.25:
            return 0.28125
        case 0.5:
            return 0.5625
        case 1:
            return 1.125
        case _:
            return x

def w_decoder(x):
    match x:
        case 0:
            return 0.125
        case 0.125:
            return 0.125
        case 0.25:
            return 0.1875
        case 0.5:
            return 0.375
        case 1:
            return 0.75
        case _:
            return 1.0


for level in levels:
    #dft = dfe[dfe['party_level'].eq(level) & dfe['2014 XP_ratio'].ge(0.30)]
    dft = dfe[dfe['party_level'].eq(level) & dfe['2014 difficulty'].isin(['Medium','Hard','Deadly'])]
    monster_cr_values = []
    for x in dft['monsters_cr']:
        monster_cr_values.extend(x)

    #fig.add_trace(go.Histogram(x=monster_cr_values, xbins_size=1, histnorm='probability', showlegend=False))

    unique_cr_values = np.unique(monster_cr_values)
    c = [np.sum(np.equal(monster_cr_values, cr))/len(monster_cr_values) for cr in unique_cr_values]
    x = [x_decoder(cr) for cr in unique_cr_values]
    w = [w_decoder(cr) for cr in unique_cr_values]
    fig.add_trace(go.Bar(
        x=x,
        y=c,
        width=w,
        showlegend=False,
        #customdata=dfC.index,
        #hovertemplate = '<b>' + g + '</b><br>'
        #    + 'CR %{customdata}<br>'
        #    + 'Monsters %{y:,.0f}'
        #    + '<extra></extra>'
    ), row=int(level), col=1)
    fig.add_trace(go.Scatter(
        x=[level,level],
        y=[0,max(c)],
        mode='lines',
        line_color='black',
        line_dash='dash',
        showlegend=False,
        hoverinfo='skip',
    ), row=int(level), col=1)


layout = {}
for level in levels: 
    if level == 1:
        layout['yaxis'] = dict(title_text='monsters (%)', tickformat='.0%', dtick=0.10, minor_dtick=0.05)
    else:
        layout[f'yaxis{level:.0f}'] = dict(title_text='monsters (%)', tickformat='.0%', dtick=0.10, minor_dtick=0.05)

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    #barmode='stack',
    xaxis20=dict(title_text='challenge rating', range=[0,30], tickformat='.0f'),
    **layout,
    width=550, 
    height=150*20,
)
for a in fig.layout.annotations:
    a.update(y=a['y']-0.006)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')

In [319]:
# Plots a histogram of monsters per encounter
import plotly.graph_objects as go

def x_decoder(x):
    match x:
        case 0:
            return 0
        case 0.125:
            return 0.125
        case 0.25:
            return 0.28125
        case 0.5:
            return 0.5625
        case 1:
            return 1.125
        case _:
            return x

def w_decoder(x):
    match x:
        case 0:
            return 0.125
        case 0.125:
            return 0.125
        case 0.25:
            return 0.1875
        case 0.5:
            return 0.375
        case 1:
            return 0.75
        case _:
            return 1.0

fig = go.Figure()

level = 5
difficulties = [
    #'Trivial',
    #'Easy',
    #'Medium',
    #'Hard',
    #'Deadly',
    'Very Deadly',
]


dft = dfe[dfe['party_level'].eq(level) & dfe['2014 difficulty'].isin(difficulties)]
monster_cr_values = []
for x in dft['monsters_cr']:
    monster_cr_values.extend(x)

#unique_cr_values = np.unique(monster_cr_values)
"""unique_cr_values = [0,1/8,1/4,1/2] + [l for l in range(1, 21)]
c = [np.sum(np.equal(monster_cr_values, cr))/len(monster_cr_values) for cr in unique_cr_values]
x = [x_decoder(cr) for cr in unique_cr_values]
w = [w_decoder(cr) for cr in unique_cr_values]
fig.add_trace(go.Bar(
    x=x,
    y=c,
    width=w,
    showlegend=False,
    customdata=np.unique(monster_cr_values),
    hovertemplate = 'CR %{customdata}<br>Monsters %{y:.2%}<extra></extra>'
))

fig.add_trace(go.Scatter(
    x=[level,level],
    y=[0,max(c)],
    mode='lines',
    line_color='black',
    line_dash='dash',
    showlegend=False,
    hoverinfo='skip',
))
"""

unique_cr_values = [0,1/8,1/4,1/2] + [l for l in range(1, 21)]
x = list(range(len(y)))
t = ['0','1/8','1/4','1/2'] + [f'{l}' for l in range(1, 21)]
y = [np.sum(np.equal(monster_cr_values, cr))/len(monster_cr_values) for cr in unique_cr_values]
ymax = np.round(1.1*max(y), 2)

fig.add_trace(go.Bar(
    x=x,
    y=y,
    showlegend=False,
    #marker_line_width=0,
    hovertemplate = 'CR %{x}<br>Monsters %{y:.2%}<extra></extra>'
))

fig.add_trace(go.Scatter(
    x=2*[t.index(f'{level}')],
    y=[0,ymax],
    mode='lines',
    line_color='black',
    line_dash='dash',
    showlegend=False,
    hoverinfo='skip',
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    #barmode='stack',
    bargap=0,
    yaxis=dict(title_text='monsters (%)', range=[0,ymax], tickformat='.0%', dtick=0.10, minor_dtick=0.05),
    #xaxis=dict(title_text='challenge rating', range=[0,16], dtick=5, minor_dtick=1, tickformat='.0f'),
    xaxis=dict(title_text='book', range=[0,18], automargin=True, tickmode='array', tickvals=x, ticktext=t),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')

In [255]:
# Plots the distribution of encounter XP ratios for each book using the 2014 rules
import plotly.graph_objects as go
import numpy as np

rules = '2014'

fig = go.Figure()

colors = iter(COLOR_LIST)
for lvl in dfe['party_level'].unique():
    dft = dfe[dfe['party_level'].eq(lvl) & dfe['2014 difficulty'].isin(['Medium','Hard','Deadly','Very Deadly'])]
    monster_cr_values = []
    for x in dft['monsters_cr']:
        monster_cr_values.extend(x)

    fig.add_trace(go.Scatter(
        x=tfb.jitter([lvl]*len(monster_cr_values), 0.10), 
        y=tfb.jitter(monster_cr_values, 0.10), 
        mode='markers',
        name=book,
        marker_size=3,
        line_color=COLOR_LIST[0],
        showlegend=False,
    ))

fig.add_trace(go.Scatter(
    x=[0,21],
    y=[0,21],
    showlegend=False,
    mode='lines',
    line_color='black',
    line_dash='dash',
    hoverinfo='skip',
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    #barmode='stack',
    xaxis=dict(title_text='level', automargin=True, range=[0,21], dtick=5, minor_dtick=1),
    yaxis=dict(title_text='challenge rating'),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')

# Stats by level

In [ ]:
# adjusted hit points by level for each difficulty
import plotly.graph_objects as go

stat = 'adj HP'

groups = [
    #{'name': 'Trivial encounters', 'difficulties': ['Trivial']},
    {'name': 'Easy encounters', 'difficulties': ['Easy']},
    {'name': 'Medium encounters', 'difficulties': ['Medium']},
    {'name': 'Hard encounters', 'difficulties': ['Hard']},
    {'name': 'Deadly encounters', 'difficulties': ['Deadly']},
    #{'name': 'Very Deadly encounters', 'difficulties': ['Very Deadly']},
]

fig = go.Figure()

colors = iter(COLOR_LIST)
for group in groups:
    dft = dfem[dfem['2014 difficulty'].isin(group['difficulties'])]
    dft = dft[['party_level',stat]].groupby(['party_level']).mean().reset_index()

    tfb.plot_data_and_fit(fig, 
        x=dft['party_level'],
        y=dft[stat],
        line_color=next(colors), 
        name=group['name'],
        legendgroup=group['name'],
        hovertemplate='level %{x:.0f}<br>' + stat + ' %{y:.1f}<extra></extra>',
    )

dfG = dfm[['CR',stat]].groupby(['CR']).mean().reset_index()
dfG = dfG[dfG['CR'].between(1,20)]
tfb.plot_data_and_fit(fig, 
    x=dfG['CR'],
    y=dfG[stat],
    line_color='black', 
    name='baseline',
    legendgroup='baseline',
    hovertemplate='<b>baseline</b><br>level %{x:.0f}<br>' + stat + ' %{y:.1f}<extra></extra>',
)

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', automargin=True, range=[0,21], dtick=5, minor_dtick=1),
    yaxis=dict(title_text=stat, range=[0,500], dtick=100, minor_dtick=20),
    legend=dict(xanchor='left', x=0.00, yanchor='top', y=1.00),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
if SAVEFIGS:
    fig.update_layout(autosize=True, width=None, height=None)
    tfb.save_fig_html(fig, format='large', name=f'./fig-monster-adj-hp-vs-level-large')
    tfb.save_fig_html(fig, format='small', name=f'./fig-monster-adj-hp-vs-level-small')

In [ ]:
# adjusted hit points by level
import plotly.graph_objects as go

stat = 'adj HP'

difficulties = ['Trivial','Easy','Medium','Hard','Deadly','Very Deadly']
levels = list(range(1,21))
values = []
for lvl in levels:
    monsters = []
    for e in encounters:
        if np.mean(e['party']) != lvl: continue
        if e['2014 difficulty'] not in difficulties: continue
        for m, cr in zip(e['monsters'], e['monsters_cr']):
            if cr > 0:
                monsters.append(monster_decoder.get(m, m))
    monsters = monsters
    if not monsters: continue
    
    dft = dfm.set_index('Monster ID')
    dft = dft[dft['eXP'].gt(0)]
    dft['book appearances'] = 0
    for monster in dft.index:
        dft.loc[monster, 'book appearances'] = len([m for m in monsters if m == monster])
    values.append(sum(dft['book appearances']*dft[stat])/sum(dft['book appearances']))


fig = go.Figure()

fig.add_trace(go.Scatter(
    x=levels,
    y=values,
    #showlegend=False,
    name='adventure monsters',
    mode='lines+markers',
))

dfG = dft[['CR',stat]].groupby(['CR']).mean().reset_index()
fig.add_trace(go.Scatter(
    x=dfG['CR'],
    y=dfG[stat],
    #showlegend=False,
    name='all monsters',
    mode='lines+markers',
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', automargin=True, range=[0,21], dtick=5, minor_dtick=1),
    yaxis=dict(title_text=stat, range=[0,500], dtick=100, minor_dtick=50),
    legend=dict(xanchor='left', x=0.00, yanchor='top', y=1.00),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-monster-adj-hp-vs-level-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-monster-adj-hp-vs-level-small')

In [ ]:
# encounter adjusted hit points by level for each difficulty
import plotly.graph_objects as go

stat = 'adj HP'

groups = [
    #{'name': 'Trivial encounters', 'difficulties': ['Trivial']},
    {'name': 'Easy encounters', 'difficulties': ['Easy'], 'XP ratio': (0.15 + 0.30)/2},
    {'name': 'Medium encounters', 'difficulties': ['Medium'], 'XP ratio': (0.30 + 0.45)/2},
    {'name': 'Hard encounters', 'difficulties': ['Hard'], 'XP ratio': (0.45 + 0.70)/2},
    {'name': 'Deadly encounters', 'difficulties': ['Deadly'], 'XP ratio': (0.70 + 1.00)/2},
    #{'name': 'Very Deadly encounters', 'difficulties': ['Very Deadly']},
]

fig = go.Figure()

dfG = dfm[['CR',stat]].groupby(['CR']).mean()

xp_baseline = (0.15 + 0.30)/2

for i in range(4):
    print('{0:.2f}'.format(np.sqrt(xp_mean[i]/xp_mean[0])))
colors = iter(COLOR_LIST)
for group in groups:
    dft = dfem[dfem['2014 difficulty'].isin(group['difficulties'])]
    dft = dft[['party_level','Encounter ID',stat]].groupby(['party_level','Encounter ID']).sum().reset_index()
    dft = dft[['party_level',stat]].groupby(['party_level']).mean()

    color = next(colors)
    tfb.plot_data_and_fit(fig, 
        x=dft.index,
        y=[dft[stat][i]/dfG[stat][i] for i in dft.index],
        line_color=color, 
        name=group['name'],
        legendgroup=group['name'],
        hovertemplate='level %{x:.0f}<br>' + stat + ' %{y:.1f}<extra></extra>',
    )
    fig.add_scatter(
        x=[1,20],
        y=2*[np.sqrt(group['XP ratio']/xp_baseline)],
        line_color=color, 
        line_dash='dash',
        showlegend=False,
        #name=group['name'],
        legendgroup=group['name'],
        hoverinfo='skip',
        #hovertemplate='level %{x:.0f}<br>' + stat + ' %{y:.1f}<extra></extra>',
    )

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', automargin=True, range=[0,21], dtick=5, minor_dtick=1),
    yaxis=dict(title_text=stat, range=[0,4], dtick=1.0, minor_dtick=0.2, tickformat='.0%'),
    legend=dict(xanchor='left', x=0.00, yanchor='top', y=1.00),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-monster-adj-hp-vs-level-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-monster-adj-hp-vs-level-small')

1.00
1.29
1.60
1.94


In [ ]:
import plotly.graph_objects as go

stat = 'adj AC'

difficulties = ['Trivial','Easy','Medium','Hard','Deadly','Very Deadly']
levels = list(range(1,21))
values = []
for lvl in levels:
    monsters = []
    for e in encounters:
        if np.mean(e['party']) != lvl: continue
        if e['2014 difficulty'] not in difficulties: continue
        for m, cr in zip(e['monsters'], e['monsters_cr']):
            if cr > 0:
                monsters.append(monster_decoder.get(m, m))
    monsters = monsters
    if not monsters: continue
    
    dft = dfm.set_index('Monster ID')
    dft = dft[dft['eXP'].gt(0)]
    dft['book appearances'] = 0
    for monster in dft.index:
        dft.loc[monster, 'book appearances'] = len([m for m in monsters if m == monster])
    values.append(sum(dft['book appearances']*dft[stat])/sum(dft['book appearances']))


fig = go.Figure()

fig.add_trace(go.Scatter(
    x=levels,
    y=values,
    #showlegend=False,
    name='adventure monsters',
    mode='lines+markers',
))

dfG = dft[['CR',stat]].groupby(['CR']).mean().reset_index()
dfG = dfG[dfG['CR'].between(1,20)]
fig.add_trace(go.Scatter(
    x=dfG['CR'],
    y=dfG[stat],
    #showlegend=False,
    name='all monsters',
    mode='lines+markers',
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', automargin=True, range=[0,21], dtick=5, minor_dtick=1),
    yaxis=dict(title_text=stat),
    legend=dict(xanchor='left', x=0.00, yanchor='top', y=1.00),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-monster-adj-ac-vs-level-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-monster-adj-ac-vs-level-small')

In [ ]:
# adjusted damage per round by level for each difficulty
import plotly.graph_objects as go

stat = 'adj DPR'

groups = [
    #{'name': 'Trivial encounters', 'difficulties': ['Trivial']},
    {'name': 'Easy encounters', 'difficulties': ['Easy']},
    {'name': 'Medium encounters', 'difficulties': ['Medium']},
    {'name': 'Hard encounters', 'difficulties': ['Hard']},
    {'name': 'Deadly encounters', 'difficulties': ['Deadly']},
    #{'name': 'Very Deadly encounters', 'difficulties': ['Very Deadly']},
]

fig = go.Figure()

colors = iter(COLOR_LIST)
for group in groups:
    dft = dfem[dfem['2014 difficulty'].isin(group['difficulties'])]
    dft = dft[['party_level',stat]].groupby(['party_level']).mean().reset_index()

    tfb.plot_data_and_fit(fig, 
        x=dft['party_level'],
        y=dft[stat],
        line_color=next(colors), 
        name=group['name'],
        legendgroup=group['name'],
        hovertemplate='level %{x:.0f}<br>' + stat + ' %{y:.1f}<extra></extra>',
    )

dfG = dfm[['CR',stat]].groupby(['CR']).mean().reset_index()
dfG = dfG[dfG['CR'].between(1,20)]
tfb.plot_data_and_fit(fig, 
    x=dfG['CR'],
    y=dfG[stat],
    line_color='black', 
    name='baseline',
    legendgroup='baseline',
    hovertemplate='level %{x:.0f}<br>' + stat + ' %{y:.1f}<extra></extra>',
)

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', automargin=True, range=[0,21], dtick=5, minor_dtick=1),
    yaxis=dict(title_text=stat, range=[0,200], dtick=50, minor_dtick=10),
    legend=dict(xanchor='left', x=0.00, yanchor='top', y=1.00),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-monster-adj-dpr-vs-level-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-monster-adj-dpr-vs-level-small')

In [ ]:
# adjusted damage per round by level
import plotly.graph_objects as go

stat = 'adj DPR'

difficulties = ['Trivial','Easy','Medium','Hard','Deadly','Very Deadly']
levels = list(range(1,21))
values = []
for lvl in levels:
    monsters = []
    for e in encounters:
        if np.mean(e['party']) != lvl: continue
        if e['2014 difficulty'] not in difficulties: continue
        for m, cr in zip(e['monsters'], e['monsters_cr']):
            if cr > 0:
                monsters.append(monster_decoder.get(m, m))
    monsters = monsters
    if not monsters: continue
    
    dft = dfm.set_index('Monster ID')
    dft = dft[dft['eXP'].gt(0)]
    dft['book appearances'] = 0
    for monster in dft.index:
        dft.loc[monster, 'book appearances'] = len([m for m in monsters if m == monster])
    values.append(sum(dft['book appearances']*dft[stat])/sum(dft['book appearances']))


fig = go.Figure()

fig.add_trace(go.Scatter(
    x=levels,
    y=values,
    #showlegend=False,
    name='adventure monsters',
    mode='lines+markers',
))

dfG = dft[['CR',stat]].groupby(['CR']).mean().reset_index()
fig.add_trace(go.Scatter(
    x=dfG['CR'],
    y=dfG[stat],
    #showlegend=False,
    name='all monsters',
    mode='lines+markers',
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', automargin=True, range=[0,21], dtick=5, minor_dtick=1),
    yaxis=dict(title_text=stat, range=[0,200], dtick=50, minor_dtick=10),
    legend=dict(xanchor='left', x=0.00, yanchor='top', y=1.00),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-monster-adj-dpr-vs-level-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-monster-adj-dpr-vs-level-small')

In [ ]:
# encounter adjusted hit points by level for each difficulty
import plotly.graph_objects as go

stat = 'adj DPR'

groups = [
    #{'name': 'Trivial encounters', 'difficulties': ['Trivial']},
    {'name': 'Easy encounters', 'difficulties': ['Easy'], 'XP ratio': (0.15 + 0.30)/2},
    {'name': 'Medium encounters', 'difficulties': ['Medium'], 'XP ratio': (0.30 + 0.45)/2},
    {'name': 'Hard encounters', 'difficulties': ['Hard'], 'XP ratio': (0.45 + 0.70)/2},
    {'name': 'Deadly encounters', 'difficulties': ['Deadly'], 'XP ratio': (0.70 + 1.00)/2},
    #{'name': 'Very Deadly encounters', 'difficulties': ['Very Deadly']},
]

fig = go.Figure()

dfG = dfm[['CR',stat]].groupby(['CR']).mean()

xp_baseline = (0.15 + 0.30)/2

for i in range(4):
    print('{0:.2f}'.format(np.sqrt(xp_mean[i]/xp_mean[0])))
colors = iter(COLOR_LIST)
for group in groups:
    dft = dfem[dfem['2014 difficulty'].isin(group['difficulties'])]
    dft = dft[['party_level','Encounter ID',stat]].groupby(['party_level','Encounter ID']).sum().reset_index()
    dft = dft[['party_level',stat]].groupby(['party_level']).mean()

    color = next(colors)
    tfb.plot_data_and_fit(fig, 
        x=dft.index,
        y=[dft[stat][i]/dfG[stat][i] for i in dft.index],
        line_color=color, 
        name=group['name'],
        legendgroup=group['name'],
        hovertemplate='level %{x:.0f}<br>' + stat + ' %{y:.1f}<extra></extra>',
    )
    fig.add_scatter(
        x=[1,20],
        y=2*[np.sqrt(group['XP ratio']/xp_baseline)],
        line_color=color, 
        line_dash='dash',
        showlegend=False,
        #name=group['name'],
        legendgroup=group['name'],
        hoverinfo='skip',
        #hovertemplate='level %{x:.0f}<br>' + stat + ' %{y:.1f}<extra></extra>',
    )

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', automargin=True, range=[0,21], dtick=5, minor_dtick=1),
    yaxis=dict(title_text=stat, range=[0,4], dtick=1.0, minor_dtick=0.2, tickformat='.0%'),
    legend=dict(xanchor='left', x=0.00, yanchor='top', y=1.00),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-monster-adj-hp-vs-level-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-monster-adj-hp-vs-level-small')

1.00
1.29
1.60
1.94


In [ ]:
import plotly.graph_objects as go

stat = 'adj AB'

difficulties = ['Trivial','Easy','Medium','Hard','Deadly','Very Deadly']
levels = list(range(1,21))
values = []
for lvl in levels:
    monsters = []
    for e in encounters:
        if np.mean(e['party']) != lvl: continue
        if e['2014 difficulty'] not in difficulties: continue
        for m, cr in zip(e['monsters'], e['monsters_cr']):
            if cr > 0:
                monsters.append(monster_decoder.get(m, m))
    monsters = monsters
    if not monsters: continue
    
    dft = dfm.set_index('Monster ID')
    dft = dft[dft['eXP'].gt(0)]
    dft['book appearances'] = 0
    for monster in dft.index:
        dft.loc[monster, 'book appearances'] = len([m for m in monsters if m == monster])
    values.append(sum(dft['book appearances']*dft[stat])/sum(dft['book appearances']))


fig = go.Figure()

fig.add_trace(go.Scatter(
    x=levels,
    y=values,
    #showlegend=False,
    name='adventure monsters',
    mode='lines+markers',
))

dfG = dft[['CR',stat]].groupby(['CR']).mean().reset_index()
dfG = dfG[dfG['CR'].between(1,20)]
fig.add_trace(go.Scatter(
    x=dfG['CR'],
    y=dfG[stat],
    #showlegend=False,
    name='all monsters',
    mode='lines+markers',
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', automargin=True, range=[0,21], dtick=5, minor_dtick=1),
    yaxis=dict(title_text=stat),
    legend=dict(xanchor='left', x=0.00, yanchor='top', y=1.00),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-monster-adj-ab-vs-level-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-monster-adj-ab-vs-level-small')

In [260]:
import plotly.graph_objects as go

stat = 'Str Save'

levels = list(range(1,21))
values = []
for lvl in levels:
    monsters = []
    for e in encounters:
        if np.mean(e['party']) != lvl: continue
        for m, cr in zip(e['monsters'], e['monsters_cr']):
            if cr > 0:
                monsters.append(monster_decoder.get(m, m))
    monsters = monsters

    dft = dfm.set_index('Monster ID')
    dft = dft[dft['eXP'].gt(0)]
    dft['book appearances'] = 0
    for monster in dft.index:
        dft.loc[monster, 'book appearances'] = len([m for m in monsters if m == monster])
    values.append(sum(dft['book appearances']*dft[stat])/sum(dft['book appearances']))


fig = go.Figure()

fig.add_trace(go.Scatter(
    x=levels,
    y=values,
    #showlegend=False,
    name='adventure monsters',
    mode='lines+markers',
))

dfG = dft[['CR',stat]].groupby(['CR']).mean().reset_index()
dfG = dfG[dfG['CR'].between(1,20)]
fig.add_trace(go.Scatter(
    x=dfG['CR'],
    y=dfG[stat],
    #showlegend=False,
    name='all monsters',
    mode='lines+markers',
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', automargin=True, range=[0,21], dtick=5, minor_dtick=1),
    yaxis=dict(title_text=stat),
    legend=dict(xanchor='left', x=0.00, yanchor='top', y=1.00),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')

In [261]:
import plotly.graph_objects as go

stat = 'Dex Save'

levels = list(range(1,21))
values = []
for lvl in levels:
    monsters = []
    for e in encounters:
        if np.mean(e['party']) != lvl: continue
        for m, cr in zip(e['monsters'], e['monsters_cr']):
            if cr > 0:
                monsters.append(monster_decoder.get(m, m))
    monsters = monsters

    dft = dfm.set_index('Monster ID')
    dft = dft[dft['eXP'].gt(0)]
    dft['book appearances'] = 0
    for monster in dft.index:
        dft.loc[monster, 'book appearances'] = len([m for m in monsters if m == monster])
    values.append(sum(dft['book appearances']*dft[stat])/sum(dft['book appearances']))


fig = go.Figure()

fig.add_trace(go.Scatter(
    x=levels,
    y=values,
    #showlegend=False,
    name='adventure monsters',
    mode='lines+markers',
))

dfG = dft[['CR',stat]].groupby(['CR']).mean().reset_index()
dfG = dfG[dfG['CR'].between(1,20)]
fig.add_trace(go.Scatter(
    x=dfG['CR'],
    y=dfG[stat],
    #showlegend=False,
    name='all monsters',
    mode='lines+markers',
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', automargin=True, range=[0,21], dtick=5, minor_dtick=1),
    yaxis=dict(title_text=stat),
    legend=dict(xanchor='left', x=0.00, yanchor='top', y=1.00),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')

In [262]:
import plotly.graph_objects as go

stat = 'Con Save'

levels = list(range(1,21))
values = []
for lvl in levels:
    monsters = []
    for e in encounters:
        if np.mean(e['party']) != lvl: continue
        for m, cr in zip(e['monsters'], e['monsters_cr']):
            if cr > 0:
                monsters.append(monster_decoder.get(m, m))
    monsters = monsters

    dft = dfm.set_index('Monster ID')
    dft = dft[dft['eXP'].gt(0)]
    dft['book appearances'] = 0
    for monster in dft.index:
        dft.loc[monster, 'book appearances'] = len([m for m in monsters if m == monster])
    values.append(sum(dft['book appearances']*dft[stat])/sum(dft['book appearances']))


fig = go.Figure()

fig.add_trace(go.Scatter(
    x=levels,
    y=values,
    #showlegend=False,
    name='adventure monsters',
    mode='lines+markers',
))

dfG = dft[['CR',stat]].groupby(['CR']).mean().reset_index()
dfG = dfG[dfG['CR'].between(1,20)]
fig.add_trace(go.Scatter(
    x=dfG['CR'],
    y=dfG[stat],
    #showlegend=False,
    name='all monsters',
    mode='lines+markers',
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', automargin=True, range=[0,21], dtick=5, minor_dtick=1),
    yaxis=dict(title_text=stat),
    legend=dict(xanchor='left', x=0.00, yanchor='top', y=1.00),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')

In [263]:
import plotly.graph_objects as go

stat = 'Wis Save'

levels = list(range(1,21))
values = []
for lvl in levels:
    monsters = []
    for e in encounters:
        if np.mean(e['party']) != lvl: continue
        for m, cr in zip(e['monsters'], e['monsters_cr']):
            if cr > 0:
                monsters.append(monster_decoder.get(m, m))
    monsters = monsters

    dft = dfm.set_index('Monster ID')
    dft = dft[dft['eXP'].gt(0)]
    dft['book appearances'] = 0
    for monster in dft.index:
        dft.loc[monster, 'book appearances'] = len([m for m in monsters if m == monster])
    values.append(sum(dft['book appearances']*dft[stat])/sum(dft['book appearances']))


fig = go.Figure()

fig.add_trace(go.Scatter(
    x=levels,
    y=values,
    #showlegend=False,
    name='adventure monsters',
    mode='lines+markers',
))

dfG = dft[['CR',stat]].groupby(['CR']).mean().reset_index()
dfG = dfG[dfG['CR'].between(1,20)]
fig.add_trace(go.Scatter(
    x=dfG['CR'],
    y=dfG[stat],
    #showlegend=False,
    name='all monsters',
    mode='lines+markers',
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', automargin=True, range=[0,21], dtick=5, minor_dtick=1),
    yaxis=dict(title_text=stat),
    legend=dict(xanchor='left', x=0.00, yanchor='top', y=1.00),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')

In [264]:
import plotly.graph_objects as go

stat = 'Int Save'

levels = list(range(1,21))
values = []
for lvl in levels:
    monsters = []
    for e in encounters:
        if np.mean(e['party']) != lvl: continue
        for m, cr in zip(e['monsters'], e['monsters_cr']):
            if cr > 0:
                monsters.append(monster_decoder.get(m, m))
    monsters = monsters

    dft = dfm.set_index('Monster ID')
    dft = dft[dft['eXP'].gt(0)]
    dft['book appearances'] = 0
    for monster in dft.index:
        dft.loc[monster, 'book appearances'] = len([m for m in monsters if m == monster])
    values.append(sum(dft['book appearances']*dft[stat])/sum(dft['book appearances']))


fig = go.Figure()

fig.add_trace(go.Scatter(
    x=levels,
    y=values,
    #showlegend=False,
    name='adventure monsters',
    mode='lines+markers',
))

dfG = dft[['CR',stat]].groupby(['CR']).mean().reset_index()
dfG = dfG[dfG['CR'].between(1,20)]
fig.add_trace(go.Scatter(
    x=dfG['CR'],
    y=dfG[stat],
    #showlegend=False,
    name='all monsters',
    mode='lines+markers',
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', automargin=True, range=[0,21], dtick=5, minor_dtick=1),
    yaxis=dict(title_text=stat),
    legend=dict(xanchor='left', x=0.00, yanchor='top', y=1.00),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')

In [265]:
import plotly.graph_objects as go

stat = 'Cha Save'

levels = list(range(1,21))
values = []
for lvl in levels:
    monsters = []
    for e in encounters:
        if np.mean(e['party']) != lvl: continue
        for m, cr in zip(e['monsters'], e['monsters_cr']):
            if cr > 0:
                monsters.append(monster_decoder.get(m, m))
    monsters = monsters

    dft = dfm.set_index('Monster ID')
    dft = dft[dft['eXP'].gt(0)]
    dft['book appearances'] = 0
    for monster in dft.index:
        dft.loc[monster, 'book appearances'] = len([m for m in monsters if m == monster])
    values.append(sum(dft['book appearances']*dft[stat])/sum(dft['book appearances']))


fig = go.Figure()

fig.add_trace(go.Scatter(
    x=levels,
    y=values,
    #showlegend=False,
    name='adventure monsters',
    mode='lines+markers',
))

dfG = dft[['CR',stat]].groupby(['CR']).mean().reset_index()
dfG = dfG[dfG['CR'].between(1,20)]
fig.add_trace(go.Scatter(
    x=dfG['CR'],
    y=dfG[stat],
    #showlegend=False,
    name='all monsters',
    mode='lines+markers',
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', automargin=True, range=[0,21], dtick=5, minor_dtick=1),
    yaxis=dict(title_text=stat),
    legend=dict(xanchor='left', x=0.00, yanchor='top', y=1.00),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')

# chance to hit

In [266]:
import plotly.graph_objects as go

def attack_hit_crit_prob(AC, AB):
    return max(0.05, min(0.95, 0.05*(21 + AB - AC)))

def pc_baseline_ab(level):
    return 4.6 + level/3

#difficulties = ['Hard','Deadly','Very Deadly']
difficulties = ['Deadly','Very Deadly']

levels = list(range(1,21))
values = []
for lvl in levels:
    monsters = []
    for e in encounters:
        if np.mean(e['party']) != lvl: continue
        if e['2014 difficulty'] not in difficulties: continue
        for m, cr in zip(e['monsters'], e['monsters_cr']):
            if cr > 0:
                monsters.append(monster_decoder.get(m, m))
    monsters = monsters

    dft = dfm.set_index('Monster ID')
    dft = dft[dft['eXP'].gt(0)]
    dft['book appearances'] = 0
    for monster in dft.index:
        dft.loc[monster, 'book appearances'] = len([m for m in monsters if m == monster])

    p_ab = pc_baseline_ab(lvl)
    p_hit = np.array([attack_hit_crit_prob(m_ac, p_ab) for m_ac in dft['adj AC']])
    values.append(sum(dft['book appearances']*p_hit)/sum(dft['book appearances']))


fig = go.Figure()

fig.add_trace(go.Scatter(
    x=levels,
    y=values,
    #showlegend=False,
    name='adventure monsters',
    mode='lines+markers',
))

dfG = dft[['CR','adj AC']].groupby(['CR']).mean().reset_index()
dfG = dfG[dfG['CR'].between(1,20)]
values = [attack_hit_crit_prob(m_ac, pc_baseline_ab(lvl)) for lvl, m_ac in zip(dfG['CR'], dfG['adj AC'])]
fig.add_trace(go.Scatter(
    x=dfG['CR'],
    y=values,
    #showlegend=False,
    name='all monsters',
    mode='lines+markers',
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', automargin=True, range=[0,21], dtick=5, minor_dtick=1),
    yaxis=dict(title_text='PC chance to hit', range=[0,1], dtick=0.2, minor_dtick=0.1, tickformat='.0%'),
    legend=dict(xanchor='left', x=0.00, yanchor='top', y=1.00),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')

In [267]:
import plotly.graph_objects as go

def attack_hit_crit_prob(AC, AB):
    return max(0.05, min(0.95, 0.05*(21 + AB - AC)))

def pc_baseline_ac(level):
    return 14.7 + level/6

#difficulties = ['Hard','Deadly','Very Deadly']
difficulties = ['Deadly','Very Deadly']
levels = list(range(1,21))
values = []
for lvl in levels:
    monsters = []
    for e in encounters:
        if np.mean(e['party']) != lvl: continue
        if e['2014 difficulty'] not in difficulties: continue
        for m, cr in zip(e['monsters'], e['monsters_cr']):
            if cr > 0:
                monsters.append(monster_decoder.get(m, m))
    monsters = monsters

    dft = dfm.set_index('Monster ID')
    dft = dft[dft['eXP'].gt(0)]
    dft['book appearances'] = 0
    for monster in dft.index:
        dft.loc[monster, 'book appearances'] = len([m for m in monsters if m == monster])

    p_ac = pc_baseline_ac(lvl)
    p_hit = np.array([attack_hit_crit_prob(p_ac, m_ab) for m_ab in dft['adj AB']])
    values.append(sum(dft['book appearances']*p_hit)/sum(dft['book appearances']))


fig = go.Figure()

fig.add_trace(go.Scatter(
    x=levels,
    y=values,
    #showlegend=False,
    name='adventure monsters',
    mode='lines+markers',
))

dfG = dft[['CR','adj AB']].groupby(['CR']).mean().reset_index()
dfG = dfG[dfG['CR'].between(1,20)]
values = [attack_hit_crit_prob(pc_baseline_ac(lvl), m_ab) for lvl, m_ab in zip(dfG['CR'], dfG['adj AB'])]
fig.add_trace(go.Scatter(
    x=dfG['CR'],
    y=values,
    #showlegend=False,
    name='all monsters',
    mode='lines+markers',
))

# set layout
fig.update_layout(
    template=tfb.FIG_TEMPLATE,
    xaxis=dict(title_text='level', automargin=True, range=[0,21], dtick=5, minor_dtick=1),
    yaxis=dict(title_text='monster chance to hit', range=[0,1], dtick=0.2, minor_dtick=0.1, tickformat='.0%'),
    legend=dict(xanchor='left', x=0.00, yanchor='top', y=1.00),
    width=550, 
    height=450,
)

fig.show(config=tfb.FIG_CONFIG)

# save figures
#if SAVEFIGS:
#    fig.update_layout(autosize=True, width=None, height=None)
#    tfb.save_fig_html(fig, format='large', name=f'./fig-dmg-daily-medium-to-hard-large')
#    tfb.save_fig_html(fig, format='small', name=f'./fig-dmg-daily-medium-to-hard-small')